# Berg et al. (2020): human supragranular glutamatergic neurons, from dandiset 000023

- Paper. Berg, Sorensen, Ting, Miller et al. (2020), "Human cortical expansion involves
  diversification and specialization of supragranular intratelencephalic-projecting neurons",
  bioRxiv, doi:10.1101/2020.03.31.018820, version 1.
- Dataset. DANDI:000023, "Patch-seq recordings from human cortex (June 2020)", version
  0.210914.1900 (doi:10.48324/dandi.000023/0.210914.1900). The paper cites no version of the
  dandiset and the claims file gives none. This is the only published version, so it is pinned;
  the draft holds the same files (checked below).
- Claims file. `claims/000023/10.1101-2020.03.31.018820.yaml`, written from the bioRxiv
  preprint, version 1 (main text, Methods, figure and Extended Data legends), and from the paper's
  code, AllenInstitute/patchseq_human_L23 at commit 3daf9f0.
- Outside inputs. The dandiset holds only the human electrophysiology, with no t-type, depth or
  specimen ID. Labels (t-type, normalized L2/3 depth, region), donor pathology scores, the mouse
  features, the morphological features, the gene-depth table and the NeuN section ROIs come from
  the code repository at commit 3daf9f0. Questions to the readers, and their answers, are in `questions/000023.yaml`; the claims
  they bear on were updated in the claims file, and this notebook follows the updated criteria.

| Claim | Data | Tested |
|---|---|---|
| features-and-labels-link | with labels | yes, readings M1 and M2 (the claims that need labels use M2) |
| cohort-cells, cohort-donors, cohort-type-counts | with labels | yes |
| fig3d-depth-by-type | with labels | yes |
| fig4e-human-ephys-anova, fig4-glp2r-sag | with labels | yes |
| fig4e-mouse-ephys-anova, edtable1-species-ephys, edfig6-classifier-mouse-ephys | with inputs | yes (mouse features from the code) |
| edfig6-classifier-human-ephys, fig5c-frem3-ephys-depth | with labels | yes |
| fig6-deep-ephys-anova, fig6c-carm1p1-upstroke-rin, fig6c-col22a1-rin-fi-sag | with labels | yes |
| fig2ef-pathology-ephys | with inputs | yes |
| fig2d-pathology-scores, edfig1-marker-correlation | elsewhere (code) | yes |
| fig4f-morph-anova, fig5b-frem3-morph-depth, fig6-deep-morph-anova | elsewhere (code) | yes |
| fig5d-frem3-gene-depth | elsewhere (code) | computed; blocked (the table is the revision's) |
| fig1a-thickness, fig1b-density, fig1c-soma-area, fig1c-soma-variation | elsewhere (code) | yes |
| fig6a-deep-dendrites, edfig6-classifier-morphoelectric | elsewhere (code) | yes (methods from the answers); fig6a blocked, its readings differ |
| fig1f-frem3-heterogeneity, fig1g-deep-types-de-genes, fig3a-platform-correlation, fig6e-deep-de-genes, fig6f-smi32 | elsewhere | blocked (no pinned or public data) |

The features are computed after the sweep QC of the Methods, as the link criterion asks; IPFX's own
sweep QC, which leaves out the Methods' bias current and bridge balance limits, is run as a variant, and
a section near the end shows how every electrophysiology outcome changes with the QC and the cell set.

The notebook took 14 minutes in its last run (26 in an earlier one, at a slower network): about 5 to
read and analyse the 318 files with IPFX, about 6 for the two electrophysiology classifiers of 1000
splits each, and about 2 for the morpho-electric classifiers (250 splits each).

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import time
import warnings
import zlib
from itertools import combinations

import h5py
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import patchseq_l23 as P

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
T0 = time.time()


def rng_for(claim):
    """Each claim's own random generator, seeded from its id."""
    return np.random.default_rng(zlib.crc32(claim.encode()))


F = P.FEATURES
UNITS = dict(adapt_mean="ratio", avg_rate_hero="Hz", downstroke_long_square="mV/ms",
             fast_trough_v_long_square_rel="mV", fi_fit_slope="Hz/pA", first_isi_hero_inv="Hz",
             input_resistance="MOhm", latency_rheo="s", peak_v_long_square_rel="mV", rheobase_i="pA",
             sag="ratio", tau="s", threshold_v_long_square="mV", trough_v_long_square_rel="mV",
             upstroke_downstroke_ratio_long_square="ratio", upstroke_long_square="mV/ms", v_baseline="mV",
             width_long_square="s")

## The dataset, its versions and the paper's code tables

In [ ]:
info = P.version_info()
versions = P._get_json(f"{P.API}/dandisets/{P.DANDISET}/versions/")["results"]
print(pd.DataFrame([dict(version=v["version"], status=v["status"], assets=v["asset_count"],
                         size_gb=round(v["size"] / 1e9, 2), modified=v["modified"][:10]) for v in versions]))
summ = info["assetsSummary"]
print("\n", info["name"], "|", info["description"])
print("related:", [(r["relation"], r["url"]) for r in info["relatedResource"]])
print("archive summary: subjects", summ["numberOfSubjects"], "| files", summ["numberOfFiles"],
      "| species", [s["name"] for s in summ["species"]], "| variables", summ["variableMeasured"],
      "| technique", [t["name"] for t in summ["measurementTechnique"]], "| approach", summ.get("approach"))
assets = P.list_assets()
draft = P.list_assets("draft")
print(f"\npinned version {P.VERSION}: {len(assets)} assets, {assets.subject_id.nunique()} subjects, "
      f"{assets['size'].sum() / 1e9:.2f} GB")
print("draft holds the same files (same blobs and paths):",
      set(zip(draft.path, draft.url)) == set(zip(assets.path, assets.url)))

In [ ]:
cell_tab = P.code_table("human_IVSCC_excitatory_L23_consolidated_0131.csv")
ephys_tab = P.code_table("human_mouse_ephys_all_0127.csv")
mouse_tab = P.code_table("mouse_IVSCC_excitatory_L23_consolidated_0129.csv")
path_tab = P.code_table("pathology_scoring.csv")
morph_h = P.code_table("All_L23_Lockdown_all_raw_features.csv")
morph_m = P.code_table("All_Mouse_Cells_Lockdown_All_raw_features.csv")
morph_names = P.code_table("morpho_feature_names.csv")
genes_depth = P.code_table("frem_genes_depth.csv")
donor_tab = P.code_table("hIVSCC Manuscript Donor Table - Extended Data Table 1.csv")
for name, t in [("human cell table", cell_tab), ("feature table (human and mouse)", ephys_tab),
                ("mouse cell table", mouse_tab), ("pathology scores", path_tab),
                ("human morphology", morph_h), ("mouse morphology", morph_m), ("gene-depth results", genes_depth),
                ("donor table", donor_tab)]:
    print(f"{name}: {t.shape[0]} rows, {t.shape[1]} columns")
print("\nhuman cell table columns:", list(cell_tab.columns))

### What one file holds

Every group and dataset of one file, with the per-sweep series collapsed to one line per kind.

In [ ]:
raw0 = P.read_bytes(assets.url[0])
f0 = h5py.File(io.BytesIO(raw0), "r")
seen = {}
def _walk(name, obj):
    parts = name.split("/")
    key = name
    for n_prefix, prefix in ((1, "acquisition"), (2, "stimulus/presentation"), (2, "processing/spikes")):
        if name.startswith(prefix + "/") and len(parts) > n_prefix:
            key = "/".join(parts[:n_prefix] + ["<sweep>"] + parts[n_prefix + 1:])
    if key not in seen:
        seen[key] = (type(obj).__name__, getattr(obj, "shape", ""), str(getattr(obj, "dtype", "")))
f0.visititems(_walk)
for k, v in seen.items():
    print(k, *v)
print("\nsweep series attributes:", dict(f0["acquisition"][list(f0["acquisition"])[10]].attrs).keys())
print("session_description:", f0["session_description"][()], "| filtering:", f0["general/intracellular_ephys/filtering"][()])
f0.close()

## Feature extraction

Each file is read whole into memory from the archive (about 70 s for the 12.4 GB at the speeds seen
while preparing) and run through IPFX: sweep QC, then IPFX's long-square analysis on the sweeps that
pass. The sweep QC is the Methods', as the link criterion asks. IPFX 2.1.2's own sweep QC applies only
the noise limits (and a Vm-delta limit); the bias current (0 +/- 100 pA) and bridge balance (below 20
MOhm and 15% of input resistance) limits of the Methods are added here (`patchseq_l23.METHODS_SWEEP_QC`),
with the input resistance of IPFX's cell QC for the 15% limit. The features under IPFX's sweep QC alone,
which the first round of this attempt used, are kept as a variant. The 18 features are read from IPFX's
output as `patchseq_l23.FEATURE_SOURCES` says. This takes about 9 minutes on four processes.

In [ ]:
t = time.time()
raw = P.extract_all(assets.url, processes=4)
cells = assets.merge(raw, on="url")
cells["donor"] = cells.cell_name.str.split(".").str[:3].str.join(".")
print(f"extracted {len(cells)} files in {(time.time() - t) / 60:.1f} min; "
      f"{cells.error.notna().sum()} gave no features")
print(pd.Series(P.FEATURE_SOURCES, name="read from IPFX as").to_string())

## Data check

In [ ]:
print("files", len(cells), "| subjects (subject_id)", cells.subject_id.nunique(),
      "| donors (from cell names)", cells.donor.nunique(), "| archive summary says",
      summ["numberOfSubjects"], "subjects")
print("subject_id in the file equals the path's:", (cells.nwb_subject_id.astype(int) == cells.subject_id).all(),
      "| session_id in the file equals the path's:", (cells.nwb_session_id.astype(int) == cells.session_id).all())
print("one donor name per subject_id:", cells.groupby("subject_id").donor.nunique().max() == 1,
      "| one subject_id per donor name:", cells.groupby("donor").subject_id.nunique().max() == 1)
per_donor = cells.groupby("donor").size()
print("files per donor: min", per_donor.min(), "median", per_donor.median(), "max", per_donor.max())
print("species", cells.species.value_counts().to_dict(), "| sex of donors",
      cells.groupby("donor").sex.first().value_counts().to_dict())
age_y = cells.groupby("donor").age.first().str.extract(r"P([\d.]+)D")[0].astype(float) / 365.25
print(f"donor age (from ISO durations in days): {age_y.min():.1f} to {age_y.max():.1f} years")

# recordings deposited more than once
print("\nduplicate cell names:", cells.cell_name.duplicated().sum(),
      "| duplicate session start times:", cells.session_start.duplicated().sum(),
      "| files with identical feature vectors:", cells.loc[cells.error.isna(), F].round(9).duplicated(keep=False).sum())

# sweeps, rates, units
sw = pd.DataFrame([dict(path=p, **s) for p, ss in zip(cells.path, cells.sweeps) for s in ss])
from ipfx.stimulus import StimulusOntology
import ipfx.json_utilities as ju
ont = StimulusOntology(ju.read(StimulusOntology.DEFAULT_STIMULUS_ONTOLOGY_FILE))


def ipfx_name(code):
    try:
        return ont.find_one(code.replace("_DA_0", ""), tag_type="code").tags(tag_type="name")[0][1]
    except Exception:
        return "not in the ontology"


names = {c: ipfx_name(c) for c in sw.stim.unique()}
sw["ipfx_name"] = sw.stim.map(names)
print("\nsweeps by kind:")
print(sw.groupby("kind").agg(n=("series", "size"), rates_hz=("rate", lambda x: sorted(set(x))),
                             unit=("unit", lambda x: sorted(set(x))),
                             conversion=("conversion", lambda x: sorted(f"{v:g}" for v in set(x)))))
print("\ncurrent-clamp stimulus sets (sweeps):")
print(sw[sw.kind == "CurrentClampSeries"].groupby(["ipfx_name", "stim"]).size().to_string())
ls_sw = sw[sw.ipfx_name == "Long Square"]
print(f"\nlong-square sweeps: {len(ls_sw)} in {ls_sw.path.nunique()} files; bridge balance "
      f"{ls_sw.bridge_balance.min():.1f} to {ls_sw.bridge_balance.max():.1f} MOhm; bias current "
      f"{ls_sw.bias_current.min():.1f} to {ls_sw.bias_current.max():.1f} pA "
      f"({ls_sw.bias_current.isna().sum()} sweeps without one)")
durs = sorted(set(d for ds in cells.ls_durations.dropna() for d in ds))
amps = [a for aa in cells.ls_amps.dropna() for a in aa]
print("long-square step durations (s):", durs, "| amplitudes", min(amps), "to", max(amps), "pA")
print("step onset after the experiment epoch start (s):", sorted(cells.step_start.dropna().round(4).unique()))
print("\nlong squares per file: recorded", cells.n_ls_sweeps.describe()[["min", "50%", "max"]].to_dict(),
      "| passing sweep QC", cells.n_ls_passed.describe()[["min", "50%", "max"]].to_dict(),
      "| total passing", int(cells.n_ls_passed.sum()), "of", int(cells.n_ls_sweeps.sum()))
print("cell QC (computed, not used to drop cells): fail", int(cells.cell_qc_failed.sum()), "of",
      int(cells.cell_qc_failed.notna().sum()))
import ipfx
from collections import Counter
G = ["ipfxqc_" + f for f in F]  # the features under IPFX's sweep QC alone
print("\nIPFX version", ipfx.__version__)
print("long squares passing IPFX's sweep QC:", int(cells.n_ls_passed_ipfx.sum()), "| passing the Methods' sweep QC "
      "(IPFX's plus bias current and bridge balance):", int(cells.n_ls_passed.sum()), "of", int(cells.n_ls_sweeps.sum()))
fail_why = sum((Counter(d) for d in cells.ls_fail_reasons.dropna()), Counter())
print("long-square sweeps failing, by reason (a sweep can have several):", dict(fail_why.most_common()))
bias = pd.Series([b for bb in cells.ls_bias_pa.dropna() for b in bb], dtype=float)
print(f"bias current of long squares, as IPFX reads it: {bias.min():.1f} to {bias.max():.1f} pA, "
      f"{int((bias.abs() > 100).sum())} outside +/-100 pA, {int(bias.isna().sum())} with no value (kept)")
print(f"input resistance from IPFX's cell QC (for the 15% bridge balance limit): {cells.rin_cell_qc.min():.0f} to "
      f"{cells.rin_cell_qc.max():.0f} MOhm, missing in {int(cells.rin_cell_qc.isna().sum())} files")
print("files with features: Methods' sweep QC", int((cells[F].notna().any(axis=1)).sum()), "| IPFX's sweep QC alone",
      int((cells[G].notna().any(axis=1)).sum()), "| files where any feature differs between the two:",
      int(((cells[F].to_numpy(float) != cells[G].to_numpy(float)) & ~(cells[F].isna().to_numpy() & cells[G].isna().to_numpy())).any(axis=1).sum()))
print("\nfiles without features, and why (long squares passing the Methods' QC and IPFX's QC alone; IPFX's error):")
cells["features_under_ipfx_qc"] = cells[G].notna().any(axis=1)
print(cells.loc[cells.error.notna(), ["cell_name", "n_ls_passed", "n_ls_passed_ipfx", "features_under_ipfx_qc", "error"]].to_string())

# ranges and units of every feature used
rng_tab = pd.DataFrame({f: dict(unit=UNITS[f], n=int(cells[f].notna().sum()), min=cells[f].min(),
                                median=cells[f].median(), max=cells[f].max()) for f in F}).T
print("\nfeature ranges (all files):")
print(rng_tab.to_string())

# clock: spikes in the step against the second before it, and IPFX against the file's stored spike times
ok = cells.error.isna()
print(f"\nupward 0 mV crossings in QC-passed long squares: {int(cells.spikes_in_step.sum())} during the "
      f"step, {int(cells.spikes_before_step.sum())} in the 1 s before it "
      f"({int((cells.spikes_before_step > 0).sum())} files with any before)")
has = cells.stored_dt_ms.notna()
print(f"rheobase sweep: IPFX threshold times against processing/spikes in {int(has.sum())} files; "
      f"same spike count in {int((cells.stored_n == cells.ipfx_n)[has].sum())}; largest time difference "
      f"{cells.stored_dt_ms.max():.3f} ms (median {cells.stored_dt_ms.median():.3f})")

## Link: features-and-labels-link

> The dandiset's recordings can be matched to the paper's per-cell table (t-type and depth) by
> specimen ID, and the 18 features computed from them agree with the paper's feature table.

The files carry no SpecimenID. The criterion, as updated after the question, names two readings of
the match and asks for both: (M1) a SpecimenID stored in the files, and (M2) the SpecimenID
recovered from the file, the smallest SpecimenID of the same donor in the code's table that is above
the file's session id, with the code's per-cell features as the check. M2 is a reconstruction that
neither the paper nor the code states. It is also checked against a match made on the features
alone.


In [ ]:
tab = cell_tab.set_index("SpecimenID")
etab = ephys_tab.set_index("specimen_id")
for k in P.DROP_NONPOSITIVE:
    etab[k] = etab[k].where(etab[k] > 0)
etab_h = etab.loc[etab.index.isin(tab.index)].dropna(how="all")  # human cells with features in the code
ids_in_files = set(cells.session_id) | set(cells.subject_id)
print("M1: files whose session or subject id is a SpecimenID of the cell table:", len(ids_in_files & set(tab.index)),
      "| of the feature table:", len(ids_in_files & set(etab.index)))
print("donors: in the files", cells.donor.nunique(), "| in the cell table", tab.donor.nunique(),
      "| file donors found in the table", cells.donor.isin(tab.donor).groupby(cells.donor).first().sum())

rows = []
for r in cells.itertuples():
    cand = np.sort(tab.index[tab.donor == r.donor].to_numpy())
    after = cand[cand > r.session_id]
    rows.append(dict(path=r.path, specimen_id=after[0] if len(after) else np.nan,
                     offset=after[0] - r.session_id if len(after) else np.nan,
                     gap_to_next=after[1] - after[0] if len(after) > 1 else np.nan, n_donor_cells=len(cand)))
cells = cells.merge(pd.DataFrame(rows), on="path")
print(f"\nM2: files with a candidate {cells.specimen_id.notna().sum()} of {len(cells)}; distinct candidates "
      f"{cells.specimen_id.nunique()}; offset {cells.offset.min():.0f} to {cells.offset.max():.0f} "
      f"(median {cells.offset.median():.0f}); next candidate of the donor a further {cells.gap_to_next.min():.0f} "
      f"or more above")
print("cell-table cells with a file:", cells.specimen_id.nunique(), "of", len(tab),
      "| the code's human cells with features that have a file:", int(etab_h.index.isin(cells.specimen_id).sum()),
      "of", len(etab_h))

# features computed here against the code's feature table, for the matched cells
m = cells[cells.specimen_id.isin(etab_h.index) & cells.error.isna()].set_index("specimen_id")
mine, ref = m[F].astype(float), etab_h.loc[m.index, F]
rel = (mine - ref).abs() / ref.abs()
both = mine.notna() & ref.notna()
agree = pd.DataFrame({"n_both": both.sum(), "median_rel_diff": rel.where(both).median(),
                      "share_within_10pct": (rel <= 0.1).where(both).mean(),
                      "share_identical": ((mine - ref).abs() <= 1e-6 * ref.abs() + 1e-12).where(both).mean(),
                      "here_not_code": (mine.notna() & ref.isna()).sum(), "code_not_here": (mine.isna() & ref.notna()).sum()})
print(f"\nmatched cells with features both here and in the code: {len(m)}")
print(agree.round(3).to_string())
n_within = int((agree.median_rel_diff <= 0.1).sum())
print("features whose median relative difference is within 10%:", n_within, "of 18")
same_cell = ((mine - ref).abs() <= 1e-6 * ref.abs() + 1e-12).where(both).mean(axis=1)
print(f"cells with every shared feature identical: {(same_cell == 1).sum()}; with at least 80% identical: "
      f"{(same_cell >= 0.8).sum()}; lowest share {same_cell.min():.2f}")

# the same comparison with the features under IPFX's sweep QC alone (the variant)
ma = cells[cells.specimen_id.isin(etab_h.index)].set_index("specimen_id")  # with the files that give no features
mine_i, ref_a = ma[G].astype(float).set_axis(F, axis=1), etab_h.loc[ma.index, F]
both_i = mine_i.notna() & ref_a.notna()
same_i = ((mine_i - ref_a).abs() <= 1e-6 * ref_a.abs() + 1e-12).where(both_i)
print(f"variant, IPFX's sweep QC alone: median relative difference within 10% for "
      f"{int(((mine_i - ref_a).abs() / ref_a.abs()).where(both_i).median().le(0.1).sum())} of 18 features; values on both sides "
      f"{int(both_i.sum().sum())} (Methods' QC {int(both.sum().sum())}); cells with every shared feature identical "
      f"{int((same_i.mean(axis=1) == 1).sum())}; the code's cells with features that have none here: "
      f"{int((ma[G].isna().all(axis=1)).sum())} (Methods' QC {int((ma[F].isna().all(axis=1)).sum())})")

# a match on the features alone, within donor
Fm = [f for f in F if f != "adapt_mean"]
agree_n = tot = 0
for r in cells[cells.error.isna()].itertuples():
    cand = etab_h[etab_h.index.isin(tab.index[tab.donor == r.donor])]
    if not len(cand) or r.specimen_id not in etab_h.index:
        continue
    d = ((cand[Fm] - pd.Series({f: getattr(r, f) for f in Fm})).abs() / cand[Fm].abs()).median(axis=1)
    tot += 1
    agree_n += int(d.idxmin() == r.specimen_id)
print(f"feature-only match within donor picks the M2 cell for {agree_n} of {tot} files whose M2 cell has features in the code")

# M2, file by file: the offset and the gap to the donor's next SpecimenID, for the cells the features check (the
# code's 283) and for the 35 they cannot (no features in the code's table)
cells["in283"] = cells.specimen_id.isin(etab_h.index)
print("\nM2 offset and gap to the next candidate, by whether the code's table has features for the cell:")
print(cells.groupby("in283").agg(files=("path", "size"), offset_min=("offset", "min"), offset_median=("offset", "median"),
                                 offset_max=("offset", "max"), gap_min=("gap_to_next", "min"),
                                 no_next_candidate=("gap_to_next", lambda x: int(x.isna().sum()))).to_string())
# checks that do not use the features: the date in the cell's sample_id (rig_YYMMDD_tube_...) against the file's
# local recording date, and, within one rig and day, tube numbers rising with recording time
lk = cells[["cell_name", "session_id", "specimen_id", "offset", "gap_to_next", "session_start", "in283"]].copy()
lk["sample_id"] = tab.loc[lk.specimen_id, "sample_id"].values
parts = lk.sample_id.str.split("_")
lk["rig"], lk["sample_date"], lk["tube"] = parts.str[0], parts.str[1], parts.str[2].astype(int)
t_utc = pd.to_datetime(lk.session_start, utc=True)
lk["local_start"] = t_utc.dt.tz_convert("America/Los_Angeles")
lk["file_date"] = lk.local_start.dt.strftime("%y%m%d")
lk["date_ok"] = lk.file_date == lk.sample_date
print("\nsample_id date equals the file's local (Seattle) recording date:",
      lk.groupby("in283").date_ok.agg(["sum", "size"]).rename(columns={"sum": "equal", "size": "files"}).to_dict("index"))
print("files where it does not:")
print(lk.loc[~lk.date_ok, ["cell_name", "local_start", "sample_id", "in283"]].to_string(index=False))
groups = viol = n_in = 0
for _, g in lk.assign(t=t_utc).groupby(["rig", "sample_date"]):
    if len(g) > 1:
        groups += 1
        n_in += len(g)
        viol += int(not (np.diff(g.sort_values("t").tube.to_numpy()) > 0).all())
multi = lk.groupby(["rig", "sample_date"]).specimen_id.transform("size") > 1
print(f"rig-days with two or more files: {groups} ({n_in} files); tube numbers out of recording order in {viol}; "
      f"of the 35, {int((multi & ~lk.in283).sum())} are in such a rig-day")
print("\nthe 35 cells the features cannot check:")
print(lk.loc[~lk.in283, ["cell_name", "session_id", "specimen_id", "offset", "gap_to_next", "sample_id", "file_date", "date_ok"]]
      .to_string(index=False))

# donor-level agreement: sex and age, against the cell table and the code's donor table
dn = cells.groupby("donor").agg(sex=("sex", "first"), age=("age", "first"))
dn["age_y"] = dn.age.str.extract(r"P([\d.]+)D")[0].astype(float) / 365.25
td = tab.groupby("donor").agg(gender=("gender", "first"), age_tab=("age", "first"))
dt = donor_tab.set_index("Donor")[["Gender", "Age (yr)"]]
dd = dn.join(td).join(dt)
dd["age_tab"] = pd.to_numeric(dd.age_tab, errors="coerce")
dd["sex_tab"] = dd.gender.str[0].str.upper().where(dd.gender.isin(["male", "female"]))
for name, sex_col, age_col in [("cell table", "sex_tab", "age_tab"), ("donor table", "Gender", "Age (yr)")]:
    ks, ka = dd[sex_col].notna(), dd[age_col].notna()
    bad_s = dd.index[ks & (dd.sex != dd[sex_col])].tolist()
    bad_a = dd.index[ka & ((dd.age_y - dd[age_col]).abs() >= 1)].tolist()
    print(f"{name}: donors listed {int(dd[sex_col].notna().sum())} (sex), {int(ka.sum())} (age); sex differs for {bad_s}; "
          f"age differs by a year or more for {bad_a}")
print(dd.loc[dd.index.isin(["H18.03.318", "H18.06.005"]), ["sex", "age_y", "gender", "age_tab", "Gender", "Age (yr)"]].round(2).to_string())
print("t-type or depth stored in the files: none (no such field in the listing above)")
link_ok = (cells.specimen_id.nunique() >= 283) and n_within == 18
print("\ncriterion under M2: at least 283 cells matched:", cells.specimen_id.nunique() >= 283,
      "| all 18 features within 10%:", n_within == 18)

**Result: reproduced under reading M2; under reading M1 no cell is linked.** No file carries a
SpecimenID, so M1 matches 0 files and would give not reproduced. Under M2 the file's session id lies
13 to 685 below one SpecimenID of the same donor, and taking that next id gives 318 distinct cells of
the 385-cell table (at least 283, as the criterion asks), including all 283 cells that have features
in the code. With the Methods' sweep QC, 261 of them have features both here and in the code; the
median relative difference is 0 for all 18 features, 128 cells agree on every shared feature and 248
on at least 80% of them, and a match on the features alone within donor picks the same cell in 261 of
261 files. With IPFX's sweep QC alone the agreement is closer (145 cells identical on every shared
feature, and 1 of the code's cells without features here against 22), so the code's table was not made
with the Methods' bias current and bridge balance limits as they are applied here. The 35 cells that
have no features in the code cannot be checked by features. Two checks that do not use features are
consistent with their links: the date in the cell's sample_id equals the file's local recording date
for all 35 (and for 279 of the 283; the other 4 started between 00:28 and 01:24, the day after the
sample's date), and in each of the 71 rig-days with two or more files (253 files, 27 of the 35 among
them) tube numbers rise with recording time without exception. Their M2 offsets (13 to 375) lie within
those of the checked cells (13 to 685). The date check separates only cells recorded on different
days, so these checks support the 35 links without proving them. At the donor level, sex differs for
one donor (H18.03.318) and age by a year or more for one (H18.06.005). The files hold no t-type or
depth.

Every claim below that needs a t-type, a depth or a region uses reading M2 of the link: the labels
come from the code's table through the recovered SpecimenID. Under reading M1 none of them could be
tested, since no cell is linked. The features are those under the Methods' sweep QC; `df_ipfx` holds
the same cells with the features under IPFX's sweep QC alone, used in the section on cell sets.

In [ ]:
# The analysis frame: every file, its matched cell's labels from the code's table, and the features computed here.
lab = tab.loc[cells.specimen_id]
df = cells.assign(tt=lab.SeuratMapping.str.split().str[-1].values, nd=lab.L23_depth_normalized.values,
                  region=lab.region.values, in283=cells.specimen_id.isin(etab_h.index).values)
df["deep_frem3"] = (df.tt == "FREM3") & (df.nd > 0.575)
HOM, DEEP = ["LTK", "GLP2R", "FREM3"], ["deep FREM3", "CARM1P1", "COL22A1"]
df["deep_group"] = np.where(df.deep_frem3, "deep FREM3", np.where(df.tt.isin(["CARM1P1", "COL22A1"]), df.tt, None))
nmiss = df[F].isna().sum(axis=1)
# the variant: the same cells with the features under IPFX's sweep QC alone (the first round's features)
df_ipfx = df.copy()
df_ipfx[F] = df[G].to_numpy()
print(df.tt.value_counts().to_string())
print("cells with a normalized depth:", int(df.nd.notna().sum()), "| regions", df.region.value_counts().to_dict())


def anova(d, group, feats=F):
    """The t-type ANOVA of the claims file for every feature, BH over the features."""
    out = pd.DataFrame([dict(feature=f, **P.ols_hc3(d, f, group)) for f in feats])
    out["p_bh"] = P.bh(out.p)
    return out.sort_values("r2", ascending=False).reset_index(drop=True)


def depth_regression(d, feats=F):
    out = pd.DataFrame([dict(feature=f, **P.ols_hc3(d, f, "nd", categorical=False)) for f in feats])
    out["r"] = np.sign(out.slope) * np.sqrt(out.r2)
    out["p_bh"] = P.bh(out.p)
    return out.sort_values("r2", ascending=False).reset_index(drop=True)

## cohort-cells

> 385 Patch-seq neurons passing transcriptomic QC mapped to the five supragranular glutamatergic
> t-types, and 283 of them had recordings complete enough to compute electrophysiological features.

In [ ]:
n385 = int(df.specimen_id.notna().sum())
print("cells of the code's 385-cell table with a recording here:", n385, "(expected 385)")
print("files without a cell of the table under M2:", int(df.specimen_id.isna().sum()),
      "(M2 gives every file whose donor is in the table a candidate, so this count cannot find other cells; "
      "the date and tube checks of the link are the evidence that the 318 are the table's cells)")
readings = {"(a) at least one feature": nmiss < 18, "(b) all 18": nmiss == 0, "(c) at most 3 of 18 missing": nmiss <= 3}
for k, v in readings.items():
    print(f"{k}: {int(v.sum())} (expected 283)")
print("\nvariant: cells with features in the code's own table:", len(etab_h), "| of them with a file here:",
      int(df.in283.sum()))
print("cells with features here but none in the code's table:", int(((nmiss < 18) & ~df.in283).sum()),
      "| of them failing cell QC:", int(((nmiss < 18) & ~df.in283 & df.cell_qc_failed.astype(bool)).sum()))
print("the code's cells with features that give none here:", int((df.in283 & (nmiss == 18)).sum()))
print("\nby donor: files per donor", df.groupby("donor").size().describe()[["min", "50%", "max"]].to_dict())
nmiss_i = df_ipfx[F].isna().sum(axis=1)
print("variant, IPFX's sweep QC alone: (a)", int((nmiss_i < 18).sum()), "(b)", int((nmiss_i == 0).sum()), "(c)", int((nmiss_i <= 3).sum()))
print("the code's cells with features that give none here: Methods' QC", int((df.in283 & (nmiss == 18)).sum()),
      "| IPFX's QC alone", int((df.in283 & (nmiss_i == 18)).sum()))
cohort_cells_match = {k: (n385 == 385, int(v.sum()) == 283) for k, v in readings.items()}
print(cohort_cells_match)

**Result: not reproduced.** 318 cells of the code's 385-cell table have a recording here, not 385,
and the cells with features number 280, 241 and 280 under readings (a), (b) and (c), not 283; with
IPFX's sweep QC alone they are 307, 256 and 307. The code's own table has features for exactly 283
cells, and all 283 have a file here; 22 of them give no features here under the Methods' sweep QC (1
under IPFX's alone), and 19 further cells give features here but have none in the code (2 of them
fail cell QC). M2 gives every file a cell of the table, so no other cell can be found among the files
and none is counted apart.

## cohort-donors

> The neurons came from neurosurgical specimens of 90 tissue donors.

In [ ]:
print("(a) subjects in the dandiset:", cells.subject_id.nunique())
print("(b) donors of the code's table cells that are here:", df.loc[df.specimen_id.notna(), "donor"].nunique())
print("for reference: donors in the code's 385-cell table", tab.donor.nunique(), "| of its 283 cells with features",
      tab.loc[etab_h.index, "donor"].nunique(), "| in the pathology table",
      path_tab.donor_id.str[:10].nunique(), "(rows", len(path_tab), ")")
print("pathology donors with a file here:", int(path_tab.donor_id.str[:10].isin(cells.donor).sum()))

**Result: not reproduced.** Both readings give 56 donors, not 90. The code's cell table has 82
donors (54 among its 283 cells with features) and the pathology table 89, of which 52 have a file
here.


## cohort-type-counts

> The electrophysiology figures show 66 LTK, 25 GLP2R and 136 FREM3 neurons, and 21 deep FREM3,
> 17 CARM1P1 and 37 COL22A1 neurons.

In [ ]:
expected_counts = pd.Series({"LTK": 66, "GLP2R": 25, "FREM3": 136, "deep FREM3": 21, "CARM1P1": 17, "COL22A1": 37})


def type_counts(d):
    return pd.Series({"LTK": (d.tt == "LTK").sum(), "GLP2R": (d.tt == "GLP2R").sum(), "FREM3": (d.tt == "FREM3").sum(),
                      "deep FREM3": d.deep_frem3.sum(), "CARM1P1": (d.tt == "CARM1P1").sum(),
                      "COL22A1": (d.tt == "COL22A1").sum()})


ct = pd.DataFrame({k: type_counts(df[v]) for k, v in readings.items()})
ct["variant: the code's 283"] = type_counts(df[df.in283])
ct["variant: IPFX's sweep QC, (a)"] = type_counts(df_ipfx[nmiss_i < 18])
ct["all files"] = type_counts(df)
ct.insert(0, "expected", expected_counts)
print(ct.to_string())
print("\nmatches per reading:", {k: int((ct[k] == ct.expected).sum()) for k in list(readings) + ["variant: the code's 283"]})
print("FREM3 cells without a depth:", int(((df.tt == "FREM3") & df.nd.isna()).sum()))

**Result: not reproduced.** At most one of the six counts matches under any reading: readings (a) and
(c) give 71 LTK, 24 GLP2R, 136 FREM3, 24 deep FREM3, 16 CARM1P1 and 33 COL22A1 (FREM3 matches), and
reading (b) gives 62, 23, 112, 16, 12 and 32 (none). The code's own 283 cells give 67, 25, 143, 25, 17
and 31 (two match), and IPFX's sweep QC alone, reading (a), gives 75, 25, 156, 28, 18 and 33 (one).

## fig3d-depth-by-type

> LTK neurons lie mainly in L2 and at the L2/L3 border, GLP2R mainly in upper L3, FREM3 span L2
> and L3 and continue into L4, and CARM1P1 and COL22A1 lie almost entirely in deep L3 and along the
> L3/L4 border.

In [ ]:
def depth_summary(d):
    g = d.dropna(subset=["nd"]).groupby("tt").nd
    return pd.DataFrame({"n": g.size(), "median": g.median(), "min": g.min(), "max": g.max(),
                         "share_deeper_0.575": g.apply(lambda x: (x > 0.575).mean()),
                         "share_deeper_1": g.apply(lambda x: (x > 1).mean())}).loc[P.TYPE_ORDER]


def fig3d_verdict(s, d):
    ltk_med = s.loc["LTK", "median"]
    fre = d.loc[(d.tt == "FREM3"), "nd"].dropna()
    parts = {"LTK median below GLP2R median": ltk_med < s.loc["GLP2R", "median"],
             "GLP2R median below 0.575": s.loc["GLP2R", "median"] < 0.575,
             "FREM3 has cells shallower than the LTK median": (fre < ltk_med).any(),
             "FREM3 has cells deeper than 1": (fre > 1).any()}
    for th in (0.9, 0.8):
        parts[f"CARM1P1 and COL22A1 at least {th:.0%} deeper than 0.575"] = bool(
            (s.loc[["CARM1P1", "COL22A1"], "share_deeper_0.575"] >= th).all())
    return parts


full = tab.assign(tt=tab.SeuratMapping.str.split().str[-1], nd=tab.L23_depth_normalized)
s_full = depth_summary(full)
print("judged, as the criterion says: every cell of the code's table with a depth, with or without features")
print(s_full.round(3).to_string())
v_full = fig3d_verdict(s_full, full)
for k, v in v_full.items():
    print(f"  {k}: {v}")
s_here = depth_summary(df)
print("\nvariant: the cells with a recording here (reading M2 of the link)")
print(s_here.round(3).to_string())
for k, v in fig3d_verdict(s_here, df).items():
    print(f"  {k}: {v}")
print("\nper donor: donors contributing CARM1P1", df[df.tt == "CARM1P1"].donor.nunique(), "| COL22A1",
      df[df.tt == "COL22A1"].donor.nunique(), "| depths range", df.nd.min().round(3), "to", df.nd.max().round(3))
fig, ax = plt.subplots(figsize=(6, 3))
for i, t in enumerate(P.TYPE_ORDER):
    y = df.loc[df.tt == t, "nd"].dropna()
    ax.scatter(np.full(len(y), i) + rng_for("fig3d-depth-by-type").uniform(-0.2, 0.2, len(y)), y, s=6)
ax.axhline(0.575, ls="--", c="k", lw=0.8)
ax.axhline(1, c="k", lw=0.8)
ax.set_xticks(range(5), P.TYPE_ORDER)
ax.invert_yaxis()
ax.set_ylabel("normalized L2/3 depth")
plt.show()
print("all", int(df.nd.notna().sum()), "points are inside the axes")

**Result: partially reproduced.** Over every cell of the code's table with a depth, as the criterion
says, the order of medians holds (LTK 0.086 below GLP2R 0.425, GLP2R below 0.575), FREM3 has cells
shallower than the LTK median, and 100% of CARM1P1 (18) and 96.7% of COL22A1 (30) cells lie deeper
than 0.575, so both readings of "almost entirely" hold. No FREM3 cell lies deeper than 1 (deepest
0.992), so "continuing into L4" does not hold. The cells with a recording here give the same (LTK
median 0.082, all 29 COL22A1 deeper than 0.575).

## fig4e-human-ephys-anova

> Among the three human homologous t-types (LTK, GLP2R, FREM3), 3 of 18 electrophysiological
> features explain more than 10% of variance by t-type (input resistance, f-I slope and rheobase),
> at FDR below 1e-5, with a maximum R2 of 0.16.

In [ ]:
NAMED_H = {"input_resistance", "fi_fit_slope", "rheobase_i"}


def fig4e_human_verdict(res):
    big = res[res.r2 > 0.1]
    exact = set(big.feature) == NAMED_H and (big.p_bh < 1e-5).all() and abs(res.r2.max() - 0.16) <= 0.016
    partial = (res[res.feature.isin(NAMED_H)].pipe(lambda r: ((r.r2 > 0.1) & (r.p_bh < 0.05)).sum()) >= 2)
    return "reproduced" if exact else ("partially_reproduced" if partial else "not_reproduced")


hom = df[df.tt.isin(HOM)]
res_h = anova(hom, "tt")
print("cells per type (with any feature):", hom[nmiss[hom.index] < 18].tt.value_counts().to_dict())
print(res_h.round(6).to_string())
print("features with R2 > 0.1:", list(res_h.feature[res_h.r2 > 0.1]), "| max R2", round(res_h.r2.max(), 4),
      "| verdict:", fig4e_human_verdict(res_h))
sup = df[df.tt.isin(["LTK", "GLP2R"]) | ((df.tt == "FREM3") & (df.nd < 0.575))]
res_sup = anova(sup, "tt")
print("\nvariant (the code's revision): superficial FREM3 only, n FREM3 =", int((sup.tt == "FREM3").sum()))
print(res_sup.head(6).round(6).to_string(), "\nverdict:", fig4e_human_verdict(res_sup))
res_283 = anova(hom[hom.in283], "tt")
print("\nvariant: the code's 283 cells only")
print(res_283.head(6).round(6).to_string(), "\nverdict:", fig4e_human_verdict(res_283))
# resampling donors, for how much the result rests on a few donors
r = rng_for("fig4e-human-ephys-anova")
donors = hom.donor.unique()
boot = []
for _ in range(200):
    pick = r.choice(donors, len(donors), replace=True)
    d = pd.concat([hom[hom.donor == x] for x in pick], ignore_index=True)
    boot.append({f: P.ols_hc3(d, f, "tt")["r2"] for f in sorted(NAMED_H)})
boot = pd.DataFrame(boot)
print(f"\n{len(donors)} donors; R2 over 200 donor resamples (2.5, 50, 97.5 percentiles):")
print(boot.quantile([0.025, 0.5, 0.975]).round(3).to_string())

**Result: partially reproduced.** Four features have R2 above 0.1, all at BH p below 1e-5: input
resistance (0.149), initial firing rate (0.137), adaptation (0.117) and rheobase (0.110). f-I slope has
0.099. The maximum R2 of 0.149 is within 0.16 +/- 0.016, but the features above 0.1 are not the three
named. Input resistance and rheobase satisfy the partial criterion. On the code's 283 cells six
features exceed 0.1 (maximum 0.170), the three named among them, so the outcome there is partial too;
with IPFX's sweep QC alone on those cells it is reproduced (section on cell sets). Over 200 donor
resamples the R2 of f-I slope ranges from 0.036 to 0.191 (2.5 to 97.5%).

## fig4e-mouse-ephys-anova

> Among the three mouse L2/3 t-types, 3 of 18 electrophysiological features explain more than
> 10% of variance (input resistance, AP width and AP downstroke rate), at FDR below 0.05, with a
> maximum R2 of 0.12, below the human maximum.

The mouse recordings are not in this dandiset; their features and t-types come from the code's tables.

In [ ]:
NAMED_M = {"input_resistance", "width_long_square", "downstroke_long_square"}
mt = mouse_tab.set_index("SpecimenID")
mt["tt"] = mt.SeuratMapping.str.split().str[-1]
mo = mt[["tt", "donor_mouse", "L23_depth_normalized"]].join(etab[F], how="inner")
mo = mo[mo[F].notna().any(axis=1)]
print("mouse cells with features:", len(mo), mo.tt.value_counts().to_dict(), "| mice", mo.donor_mouse.nunique())
res_m = anova(mo, "tt")
print(res_m.round(6).to_string())
big = res_m[res_m.r2 > 0.1]
named_ok = res_m[res_m.feature.isin(NAMED_M)].pipe(lambda r: ((r.r2 > 0.1) & (r.p_bh < 0.05)).sum())
exact = set(big.feature) == NAMED_M and (big.p_bh < 0.05).all() and abs(res_m.r2.max() - 0.12) <= 0.012 \
    and res_m.r2.max() < res_h.r2.max()
print("features with R2 > 0.1:", list(big.feature), "| max R2", round(res_m.r2.max(), 4), "| human max",
      round(res_h.r2.max(), 4), "| named features with R2 > 0.1 at FDR < 0.05:", int(named_ok),
      "| verdict:", "reproduced" if exact else ("partially_reproduced" if named_ok >= 2 else "not_reproduced"))

**Result: reproduced (on the code's mouse features).** Exactly downstroke (0.123), input resistance
(0.118) and AP width (0.116) have R2 above 0.1, each at BH p below 0.013, with a maximum of 0.123,
below the human maximum of 0.149, over 120 mouse cells (60 Agmat, 51 Rrad, 9 Adamts2).

## fig4-glp2r-sag

> GLP2R neurons, like LTK neurons, lack firing rate adaptation and have higher input resistance,
> but unlike LTK neurons (which have no sag) they have pronounced sag.

In [ ]:
pw = pd.concat([P.pairwise_mw(hom, f, "tt", HOM) for f in ["sag", "adapt_mean", "input_resistance"]])
print(pw.round(5).to_string(index=False))
sag_lg = pw[(pw.feature == "sag") & (pw.a == "LTK") & (pw.b == "GLP2R")].iloc[0]
ad_lg = pw[(pw.feature == "adapt_mean") & (pw.a == "LTK") & (pw.b == "GLP2R")].iloc[0]
print(f"\nsag: GLP2R median {sag_lg.median_b:.4f} against LTK {sag_lg.median_a:.4f}, BH p {sag_lg.p_bh:.2g}")
print(f"adaptation: GLP2R median {ad_lg.median_b:.4f} against LTK {ad_lg.median_a:.4f}, BH p {ad_lg.p_bh:.2g}")
pw283 = pd.concat([P.pairwise_mw(hom[hom.in283], f, "tt", HOM) for f in ["sag", "adapt_mean"]])
print("\nvariant: the code's 283 cells")
print(pw283[(pw283.a == "LTK") & (pw283.b == "GLP2R")].round(5).to_string(index=False))

**Result: reproduced.** GLP2R sag (median 0.054) is above LTK sag (0.025), BH p 0.046, and adaptation
does not differ between them (0.060 against 0.057, BH p 0.37). GLP2R input resistance (128 MOhm) is
above FREM3 (90, BH p 0.00001) and not different from LTK (122, BH p 0.38). The code's 283 cells give
the same (sag BH p 0.049). The sag result is close to the threshold: with cell QC applied its BH p is
0.060 (section on cell sets).

## edtable1-species-ephys

> Between human and mouse homologous t-types, 17 of 18 electrophysiological features differ at FDR
> below 0.05.

In [ ]:
def mw_scipy14(x, y):
    """scipy 1.4's mannwhitneyu default (alternative None): half the two-sided asymptotic p."""
    return stats.mannwhitneyu(x, y, alternative="two-sided", use_continuity=True, method="asymptotic").pvalue / 2


def cohen_d(x, y):
    return (np.mean(x) - np.mean(y)) / np.sqrt((np.var(x, ddof=1) + np.var(y, ddof=1)) / 2)


rows = []
for f in F:
    x, y = hom[f].dropna(), mo[f].dropna()
    rows.append(dict(feature=f, n_human=len(x), n_mouse=len(y), student=stats.ttest_ind(x, y).pvalue,
                     welch=stats.ttest_ind(x, y, equal_var=False).pvalue, d=cohen_d(x, y)))
sp = pd.DataFrame(rows)
sp["student_bh"], sp["welch_bh"] = P.bh(sp.student), P.bh(sp.welch)
hx, my = hom[hom[F].notna().all(axis=1)], mo[mo[F].notna().all(axis=1)]
sp["code_mw_bh"] = P.bh([mw_scipy14(hx[f], my[f]) for f in F])
print(sp.round(6).to_string(index=False))
n_student, n_welch, n_code = (sp.student_bh < 0.05).sum(), (sp.welch_bh < 0.05).sum(), (sp.code_mw_bh < 0.05).sum()
print(f"\nfeatures at BH p < 0.05: Student {n_student}, Welch {n_welch} (expected 17); variant, the code's "
      f"Mann-Whitney on {len(hx)} human and {len(my)} mouse cells with all 18 features: {n_code}")
print("not significant: Student", list(sp.feature[sp.student_bh >= 0.05]), "| Welch", list(sp.feature[sp.welch_bh >= 0.05]),
      "| code", list(sp.feature[sp.code_mw_bh >= 0.05]))

**Result: reproduced (with the code's mouse features).** Student's and Welch's t-tests both find 17
of 18 features different at BH p below 0.05, AP width being the exception (BH p 0.46 and 0.47).
The code's Mann-Whitney test also gives 17, with AP width the exception.

## edfig6-classifier-human-ephys and edfig6-classifier-mouse-ephys

> A random forest predicts the homologous human t-type from electrophysiological features with 67%
> accuracy. The same classifier predicts mouse L2/3 t-types with 54% accuracy, lower than for human.

1000 stratified 70/30 splits with a 600-tree forest each, on four processes (about 7 minutes).
Shuffled labels are run on 100 splits, as a reference only.

In [ ]:
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def _one_split(X, y, seed, shuffle):
    r = np.random.default_rng(seed)
    yy = r.permutation(y) if shuffle else y
    Xtr, Xte, ytr, yte = train_test_split(X, yy, test_size=0.3, stratify=yy, random_state=int(seed % 2**31))
    clf = RandomForestClassifier(n_estimators=600, class_weight="balanced", max_features="sqrt",
                                 min_samples_leaf=2, n_jobs=1, random_state=int(seed % 2**31))
    return float((clf.fit(Xtr, ytr).predict(Xte) == yte).mean())


def rf_accuracy(d, claim, n_splits=1000, shuffle=False, feats=F):
    """Mean test accuracy over stratified 70/30 splits, features standardized over all cells.

    Cells with a missing electrophysiological feature are dropped (the code drops them); `feats` are the
    columns used."""
    d = d[d[F].notna().all(axis=1) & d[feats].notna().all(axis=1)]
    X = StandardScaler().fit_transform(d[feats].to_numpy(float))
    y = d.tt.to_numpy()
    seeds = rng_for(claim + ("-shuffled" if shuffle else "")).integers(0, 2**63 - 1, n_splits)
    acc = np.array(Parallel(n_jobs=4)(delayed(_one_split)(X, y, s, shuffle) for s in seeds))
    return acc, d.tt.value_counts()

In [ ]:
t = time.time()
acc_h, n_h = rf_accuracy(hom, "edfig6-classifier-human-ephys")
acc_h0, _ = rf_accuracy(hom, "edfig6-classifier-human-ephys", n_splits=100, shuffle=True)
print("human cells with all 18 features:", n_h.to_dict(), f"| chance by the largest class {n_h.max() / n_h.sum():.3f}")
print(f"mean accuracy over {len(acc_h)} splits {acc_h.mean():.4f} (SD {acc_h.std():.4f}, 2.5-97.5% "
      f"{np.quantile(acc_h, 0.025):.3f}-{np.quantile(acc_h, 0.975):.3f}); shuffled labels {acc_h0.mean():.4f}")
acc_h283, n_h283 = rf_accuracy(hom[hom.in283], "edfig6-classifier-human-ephys-283", n_splits=100)
print(f"variant, the code's 283 cells ({n_h283.sum()} with all 18), 100 splits: {acc_h283.mean():.4f}")
print(f"{(time.time() - t) / 60:.1f} min")

**Result: reproduced.** The mean test accuracy over 1000 splits is 0.622 (SD 0.056), within
0.67 +/- 0.067, against 0.441 with shuffled labels and 0.569 for always guessing FREM3, so the margin
over the largest class is small. The code's 283 cells give 0.630 over 100 splits.

In [ ]:
t = time.time()
acc_m, n_m = rf_accuracy(mo, "edfig6-classifier-mouse-ephys")
acc_m0, _ = rf_accuracy(mo, "edfig6-classifier-mouse-ephys", n_splits=100, shuffle=True)
print("mouse cells with all 18 features:", n_m.to_dict(), f"| chance by the largest class {n_m.max() / n_m.sum():.3f}")
print(f"mean accuracy over {len(acc_m)} splits {acc_m.mean():.4f} (SD {acc_m.std():.4f}); shuffled labels "
      f"{acc_m0.mean():.4f}; human {acc_h.mean():.4f}")
print(f"{(time.time() - t) / 60:.1f} min")

**Result: reproduced (with the code's mouse features).** The mean accuracy over 1000 splits is
0.571 (SD 0.071), within 0.54 +/- 0.054 and below the human 0.622, against 0.421 with shuffled
labels and 0.491 for always guessing Agmat.

## fig5c-frem3-ephys-depth

> In FREM3 neurons, 9 of 18 electrophysiological features correlate with normalized depth at FDR
> below 0.05 (7 at FDR below 0.01); the strongest are increasing sag, increasing AP
> upstroke/downstroke ratio and decreasing latency at rheobase.

In [ ]:
def fig5c_verdict(res):
    n05, n01 = (res.p_bh < 0.05).sum(), (res.p_bh < 0.01).sum()
    top = res.head(3)
    want = {"sag": 1, "upstroke_downstroke_ratio_long_square": 1, "latency_rheo": -1}
    top_ok = set(top.feature) == set(want) and all(np.sign(top.set_index("feature").slope[k]) == v for k, v in want.items()) \
        and (top.p_bh < 1e-7).all()
    three = res.set_index("feature").loc[list(want)]
    signs = all(np.sign(three.slope[k]) == v for k, v in want.items()) and (three.p_bh < 0.05).all()
    counts_ok = abs(n05 - 9) <= 0.9 and abs(n01 - 7) <= 0.7
    return n05, n01, ("reproduced" if counts_ok and top_ok else ("partially_reproduced" if signs else "not_reproduced"))


fre = df[(df.tt == "FREM3") & df.nd.notna()]
res_d = depth_regression(fre)
print("FREM3 cells with a depth:", len(fre), "| donors", fre.donor.nunique())
print(res_d.round(8).to_string())
print("counts at FDR < 0.05, < 0.01, verdict:", fig5c_verdict(res_d))
print("top three, BH p:", {f: f"{p:.4g}" for f, p in zip(res_d.feature[:3], res_d.p_bh[:3])})
res_d283 = depth_regression(fre[fre.in283])
print("\nvariant, the code's 283 cells:", res_d283.head(4)[["feature", "n", "r2", "slope", "p_bh"]].round(8).to_string(),
      "\n", fig5c_verdict(res_d283))

**Result: partially reproduced.** Over 146 FREM3 cells with a depth from 40 donors (125 with
features), 8 features correlate with depth at BH p below 0.05 (outside 9 +/- 0.9) and 7 below 0.01 (as
expected). The three largest R2 are sag (0.487, increasing), AP upstroke/downstroke ratio (0.252,
increasing) and latency at rheobase (0.246, decreasing), as the claim states, but the ratio's BH p is
4.4e-07, not below 1e-7. The code's 283 cells give 8 and 6, and IPFX's sweep QC alone 10 and 7.

## fig6-deep-ephys-anova

> The three deep L3 types (deep FREM3, CARM1P1, COL22A1) differ in 15 of 18 electrophysiological
> features (FDR below 0.05), with R2 above 0.4 for 4 of them, while no feature reaches R2 0.4 among
> the three homologous types.

In [ ]:
deep = df[df.deep_group.notna()]
res_deep = anova(deep, "deep_group")
n05, n4 = int((res_deep.p_bh < 0.05).sum()), int((res_deep.r2 > 0.4).sum())
print("cells per group (any feature):", deep[nmiss[deep.index] < 18].deep_group.value_counts().to_dict())
print(res_deep.round(6).to_string())
print(f"features at BH p < 0.05: {n05} (expected 15 +/- 1.5); R2 > 0.4: {n4} (expected 4 +/- 0.4); "
      f"homologous-type features with R2 > 0.4: {int((res_h.r2 > 0.4).sum())} (max {res_h.r2.max():.3f})")
res_deep283 = anova(deep[deep.in283], "deep_group")
print(f"variant, the code's 283 cells: {int((res_deep283.p_bh < 0.05).sum())} at BH p < 0.05, "
      f"{int((res_deep283.r2 > 0.4).sum())} with R2 > 0.4")

**Result: reproduced.** Over 24 deep FREM3, 16 CARM1P1 and 33 COL22A1 cells, 14 features differ at BH
p below 0.05, within 15 +/- 1.5, and 4 have R2 above 0.4 (input resistance 0.600, AP width 0.525,
rheobase 0.478 and downstroke 0.474), as expected; no homologous-type feature reaches 0.4 (maximum
0.149). The code's 283 cells give 14 and 4 too. With IPFX's sweep QC alone, as in the first round, the
counts are 15 and 5 (partially reproduced), and with cell QC applied 12 and 4 (partially reproduced;
section on cell sets).

## fig6c-carm1p1-upstroke-rin and fig6c-col22a1-rin-fi-sag

> CARM1P1 neurons have a faster AP upstroke than deep FREM3 and COL22A1 neurons, and a higher
> input resistance than deep FREM3 neurons.
>
> COL22A1 neurons have very high input resistance and a steeper f-I gain than the other deep L3
> types, and less sag than CARM1P1 neurons.

In [ ]:
pwd = pd.concat([P.pairwise_mw(deep, f, "deep_group", DEEP) for f in
                 ["upstroke_long_square", "upstroke_downstroke_ratio_long_square", "input_resistance", "fi_fit_slope", "sag"]])
print(pwd.round(5).to_string(index=False))


def higher(feature, a, b):
    """True when group a is significantly above group b (BH p < 0.05 and a larger median)."""
    r = pwd[(pwd.feature == feature) & (((pwd.a == a) & (pwd.b == b)) | ((pwd.a == b) & (pwd.b == a)))].iloc[0]
    med_a, med_b = (r.median_a, r.median_b) if r.a == a else (r.median_b, r.median_a)
    return bool(r.p_bh < 0.05 and med_a > med_b)


up = {f: higher(f, "CARM1P1", "deep FREM3") and higher(f, "CARM1P1", "COL22A1")
      for f in ["upstroke_long_square", "upstroke_downstroke_ratio_long_square"]}
rin = higher("input_resistance", "CARM1P1", "deep FREM3")
print("\nCARM1P1 upstroke above both, (a) rate, (b) ratio:", up, "| CARM1P1 Rin above deep FREM3:", rin)

**Result: blocked, methods underspecified.** CARM1P1 input resistance (123 MOhm) is above deep FREM3
(86, BH p 0.0017). CARM1P1 upstroke rate (371 mV/ms) is above deep FREM3 (330, BH p 0.017) and COL22A1
(298, BH p 0.0001), so reading (a) reproduces the claim; but the upstroke/downstroke ratio of CARM1P1
(4.63) is not above that of COL22A1 (4.83, BH p 0.054), so reading (b) gives only the input resistance
part. The two readings give different outcomes under every cell set of the section on cell sets.

In [ ]:
col = {"input_resistance": higher("input_resistance", "COL22A1", "deep FREM3") and higher("input_resistance", "COL22A1", "CARM1P1"),
       "fi_fit_slope": higher("fi_fit_slope", "COL22A1", "deep FREM3") and higher("fi_fit_slope", "COL22A1", "CARM1P1"),
       "sag below CARM1P1": higher("sag", "CARM1P1", "COL22A1")}
print(col, "| parts holding:", sum(col.values()), "of 3")
pwd283 = pd.concat([P.pairwise_mw(deep[deep.in283], f, "deep_group", DEEP) for f in
                    ["upstroke_long_square", "upstroke_downstroke_ratio_long_square", "input_resistance", "fi_fit_slope", "sag"]])
print("\nvariant, the code's 283 cells:")
print(pwd283.round(5).to_string(index=False))

**Result: reproduced.** COL22A1 input resistance (297 MOhm) and f-I slope (0.170 Hz/pA) are each
above deep FREM3 (86 MOhm, 0.050) and CARM1P1 (123 MOhm, 0.117), at BH p 0.005 or less, and COL22A1 sag
(0.069) is below CARM1P1 (0.134, BH p 0.00003). All three parts hold under every cell set of the
section on cell sets.

## fig2ef-pathology-ephys

> Of 18 electrophysiological features compared between neurons from donors with low (0 to 1) and
> high (1 to 3) GFAP or IBA1 scores, only resting membrane potential under IBA1 differs, neurons
> from high-IBA1 donors being about 5 mV more hyperpolarized; the other 17 IBA1 and all 18 GFAP
> comparisons do not differ.

Donor scores come from the code's `pathology_scoring.csv`, matched on the first 10 characters of
its donor id, as the code does.

In [ ]:
pt = path_tab.assign(donor=path_tab.donor_id.str[:10])
print("pathology rows", len(pt), "| distinct donors", pt.donor.nunique())
dup = pt[pt.donor.duplicated(keep=False)]
print("donor with two rows:", dup[["donor_id", "GFAP Cortex", "Iba1 Cortex"]].to_string(index=False),
      "\n  its files here:", int(df.donor.isin(dup.donor).sum()),
      "| in both the low and the high group of a marker:",
      {m_: bool((dup[c_] <= 1).any() and (dup[c_] > 1).any()) for m_, c_ in [("GFAP", "GFAP Cortex"), ("IBA1", "Iba1 Cortex")]})


def pathology_test(d, feats_src, two_sided=False):
    out = {}
    for marker, col in [("GFAP", "GFAP Cortex"), ("IBA1", "Iba1 Cortex")]:
        low_d, high_d = set(pt.donor[pt[col] <= 1]), set(pt.donor[pt[col] > 1])
        lo, hi = feats_src[d.donor.isin(low_d).values], feats_src[d.donor.isin(high_d).values]
        rows = []
        for f in F:
            x, y = lo[f].dropna(), hi[f].dropna()
            p = (stats.mannwhitneyu(x, y, alternative="two-sided").pvalue if two_sided else mw_scipy14(x, y))
            rows.append(dict(feature=f, p=p, sig=p < 0.05 / 18, n_low=len(x), n_high=len(y),
                             median_diff=y.median() - x.median(), mean_diff=y.mean() - x.mean()))
        out[marker] = pd.DataFrame(rows)
    return out


def pathology_verdict(out):
    g, i = out["GFAP"], out["IBA1"]
    only_rmp = (not g.sig.any()) and i.sig.sum() == 1 and bool(i.set_index("feature").sig["v_baseline"])
    rmp = i.set_index("feature").loc["v_baseline"]
    size = [abs(rmp.median_diff - (-5)) <= 0.5, abs(rmp.mean_diff - (-5)) <= 0.5]
    if not only_rmp:
        return "not_reproduced"
    if all(size):
        return "reproduced"
    return "partially_reproduced" if not any(size) else "methods_underspecified (size)"


all18 = df[nmiss == 0]
for name, d in [("(a) temporal", all18[all18.region == "temporal"]), ("(b) all regions", all18)]:
    out = pathology_test(d, d)
    print(f"\n{name}: {len(d)} cells with all 18 features")
    for marker, r in out.items():
        print(f"  {marker}: low {r.n_low.max()} high {r.n_high.max()} cells; significant at 0.05/18: "
              f"{list(r.feature[r.sig])}; smallest p {r.p.min():.2g} ({r.feature[r.p.idxmin()]})")
    rmp = out["IBA1"].set_index("feature").loc["v_baseline"]
    print(f"  RMP, high minus low IBA1: medians {rmp.median_diff:.2f} mV, means {rmp.mean_diff:.2f} mV, p {rmp.p:.2g}")
    print("  verdict:", pathology_verdict(out))
    out2 = pathology_test(d, d, two_sided=True)
    print("  variant, two-sided test: significant GFAP", list(out2["GFAP"].feature[out2["GFAP"].sig]),
          "IBA1", list(out2["IBA1"].feature[out2["IBA1"].sig]))
# the updated notebook: temporal cells of the code's table with all 18 features in the code's own table
ec = etab.loc[etab.index.isin(tab.index)].dropna()
d_upd = tab.loc[ec.index].assign(donor=tab.loc[ec.index].donor)
d_upd = d_upd[d_upd.region == "temporal"]
out_u = pathology_test(d_upd, ec.loc[d_upd.index])
print(f"\nvariant, the updated notebook's procedure on the code's own features ({len(d_upd)} cells): "
      f"GFAP {list(out_u['GFAP'].feature[out_u['GFAP'].sig])} (low {out_u['GFAP'].n_low.max()}, high {out_u['GFAP'].n_high.max()}), "
      f"IBA1 {list(out_u['IBA1'].feature[out_u['IBA1'].sig])} (low {out_u['IBA1'].n_low.max()}, high {out_u['IBA1'].n_high.max()})")

**Result: partially reproduced.** Under reading (a), temporal cells (224 with all 18 features), RMP
under IBA1 is the only comparison of 36 below 0.05/18 (p 0.00089), with high-IBA1 cells more
hyperpolarized, but by 4.28 mV (medians) and 4.16 mV (means), outside 5 +/- 0.5. Under reading (b),
all regions, latency under IBA1 is significant too. Neither reading gives the paper's group sizes
(reading (a) has 73 and 139 GFAP, 187 and 29 IBA1 cells, against 60 and 125, 170 and 15). The donor
with two rows in the pathology table (H17.06.013, low in one row and high in the other for both
markers) has no file here. The updated notebook's procedure on the code's own features finds latency
and RMP under IBA1, and so do the code's 283 cells here under reading (a) (section on cell sets).

## How the electrophysiology outcomes depend on the cell set and the QC

Three choices bear on every claim above: the sweep QC (the Methods', the main reading, or IPFX's alone);
whether the 35 cells that have no features in the code's table are kept (their link rests on the date
and tube checks of the link, not on the features); and whether the cell QC of the Methods (seal, access
resistance), which the claims file does not ask for, is applied. Each claim's verdict function above is
run on each cell set. The classifier is run on 50 splits per set, as a reference.


In [ ]:
def ephys_outcomes(d):
    """The verdict of each electrophysiology claim above, recomputed on the cell set d."""
    o = {}
    nm_ = d[F].isna().sum(axis=1)
    h_ = d[d.tt.isin(HOM)]
    rh = anova(h_, "tt")
    o["fig4e-human-ephys-anova"] = (fig4e_human_verdict(rh), f"R2 > 0.1: {', '.join(rh.feature[rh.r2 > 0.1])}; max {rh.r2.max():.3f}")
    pw_ = pd.concat([P.pairwise_mw(h_, f, "tt", HOM) for f in ["sag", "adapt_mean"]]).set_index(["feature", "a", "b"])
    sg, ad = pw_.loc[("sag", "LTK", "GLP2R")], pw_.loc[("adapt_mean", "LTK", "GLP2R")]
    sag_up = sg.p_bh < 0.05 and sg.median_b > sg.median_a
    o["fig4-glp2r-sag"] = ("reproduced" if sag_up and ad.p_bh >= 0.05 else ("partially_reproduced" if sag_up else "not_reproduced"),
                          f"sag BH p {sg.p_bh:.3g}, adaptation BH p {ad.p_bh:.3g}")
    pv = pd.DataFrame([dict(s=stats.ttest_ind(h_[f].dropna(), mo[f].dropna()).pvalue,
                            w=stats.ttest_ind(h_[f].dropna(), mo[f].dropna(), equal_var=False).pvalue) for f in F])
    ns, nw = int((P.bh(pv.s) < 0.05).sum()), int((P.bh(pv.w) < 0.05).sum())
    o["edtable1-species-ephys"] = ("reproduced" if ns == nw == 17 else ("methods_underspecified" if (ns == 17) != (nw == 17) else "not_reproduced"),
                                  f"Student {ns}, Welch {nw}")
    n05_, n01_, v_ = fig5c_verdict(depth_regression(d[(d.tt == "FREM3") & d.nd.notna()]))
    o["fig5c-frem3-ephys-depth"] = (v_, f"{n05_} at FDR < 0.05, {n01_} at FDR < 0.01")
    dp = d[d.deep_group.notna()]
    rd = anova(dp, "deep_group")
    a_, b_ = int((rd.p_bh < 0.05).sum()), int((rd.r2 > 0.4).sum())
    ca, cb = abs(a_ - 15) <= 1.5, abs(b_ - 4) <= 0.4
    o["fig6-deep-ephys-anova"] = ("reproduced" if ca and cb and not (rh.r2 > 0.4).any() else ("partially_reproduced" if ca or cb else "not_reproduced"),
                                 f"{a_} at FDR < 0.05, {b_} with R2 > 0.4")
    pwd_ = pd.concat([P.pairwise_mw(dp, f, "deep_group", DEEP) for f in
                      ["upstroke_long_square", "upstroke_downstroke_ratio_long_square", "input_resistance", "fi_fit_slope", "sag"]])

    def hi(f, x, y):
        r = pwd_[(pwd_.feature == f) & (((pwd_.a == x) & (pwd_.b == y)) | ((pwd_.a == y) & (pwd_.b == x)))].iloc[0]
        mx, my = (r.median_a, r.median_b) if r.a == x else (r.median_b, r.median_a)
        return bool(r.p_bh < 0.05 and mx > my)
    up_ = [hi(f, "CARM1P1", "deep FREM3") and hi(f, "CARM1P1", "COL22A1") for f in ["upstroke_long_square", "upstroke_downstroke_ratio_long_square"]]
    rin_ = hi("input_resistance", "CARM1P1", "deep FREM3")
    vv = ["reproduced" if u and rin_ else ("partially_reproduced" if u or rin_ else "not_reproduced") for u in up_]
    o["fig6c-carm1p1-upstroke-rin"] = (vv[0] if vv[0] == vv[1] else "methods_underspecified",
                                      f"upstroke above both: (a) {up_[0]}, (b) {up_[1]}; Rin above deep FREM3: {rin_}")
    parts_ = [hi("input_resistance", "COL22A1", "deep FREM3") and hi("input_resistance", "COL22A1", "CARM1P1"),
              hi("fi_fit_slope", "COL22A1", "deep FREM3") and hi("fi_fit_slope", "COL22A1", "CARM1P1"),
              hi("sag", "CARM1P1", "COL22A1")]
    o["fig6c-col22a1-rin-fi-sag"] = ("reproduced" if all(parts_) else ("partially_reproduced" if sum(parts_) >= 2 else "not_reproduced"),
                                    f"{sum(parts_)} of 3 parts")
    a18 = d[nm_ == 0]
    order = ["reproduced", "partially_reproduced", "methods_underspecified (size)", "not_reproduced"]
    vs, txt = [], []
    for name, dd in [("(a)", a18[a18.region == "temporal"]), ("(b)", a18)]:
        out = pathology_test(dd, dd)
        vs.append(pathology_verdict(out))
        r_ = out["IBA1"].set_index("feature").loc["v_baseline"]
        txt.append(f"{name} GFAP {list(out['GFAP'].feature[out['GFAP'].sig])}, IBA1 {list(out['IBA1'].feature[out['IBA1'].sig])}, "
                   f"RMP {r_.median_diff:.2f} mV (medians)")
    o["fig2ef-pathology-ephys"] = (min(vs, key=order.index), "; ".join(txt))
    return o


cq = ~df.cell_qc_failed.astype(bool)
cell_sets = {
    "main: Methods' sweep QC, all linked cells": df,
    "Methods' sweep QC, the code's 283 only": df[df.in283],
    "Methods' sweep QC, cell QC passed": df[cq],
    "Methods' sweep QC, cell QC passed, the code's 283": df[cq & df.in283],
    "IPFX's sweep QC (round 1), all linked cells": df_ipfx,
    "IPFX's sweep QC, the code's 283 only": df_ipfx[df_ipfx.in283],
    "IPFX's sweep QC, cell QC passed": df_ipfx[cq],
}
print("cells failing cell QC: of the code's 283", int((~cq & df.in283).sum()), "| of the 35 others", int((~cq & ~df.in283).sum()))
print("cell QC failures by criterion:", df.loc[~cq, "cell_qc_tags"].str.split("; ").explode().str.split(" ").str[0].value_counts().to_dict())
sens = {k: ephys_outcomes(v) for k, v in cell_sets.items()}
claims_ = list(sens[next(iter(sens))])
print("\nverdicts (rows: cell sets; columns: claims)")
print(pd.DataFrame({c_: {k: v[c_][0] for k, v in sens.items()} for c_ in claims_}).T.to_string())
print("\nthe numbers behind them")
for c_ in claims_:
    print(c_)
    for k, v in sens.items():
        print(f"   {k}: {v[c_][1]}")
print("\nedfig6-classifier-human-ephys, mean accuracy over 50 splits (expected 0.67 +/- 0.067):")
for k in ["Methods' sweep QC, the code's 283 only", "Methods' sweep QC, cell QC passed", "IPFX's sweep QC (round 1), all linked cells"]:
    hh = cell_sets[k][cell_sets[k].tt.isin(HOM)]
    a_, n_ = rf_accuracy(hh, "edfig6-classifier-human-ephys-" + k, n_splits=50)
    print(f"   {k}: {a_.mean():.4f} ({n_.sum()} cells with all 18 features)")


**Result: the outcomes above hold for the main cell set; three choices move some of them.** Leaving
out the 35 cells without features in the code changes one outcome under the Methods' sweep QC:
fig2ef-pathology-ephys becomes not reproduced, as latency under IBA1 is significant too. With IPFX's
sweep QC alone (the first round), fig6-deep-ephys-anova is partially reproduced (15 and 5), and on the
code's 283 cells fig4e-human-ephys-anova is reproduced (maximum R2 0.161) and edtable1-species-ephys
methods underspecified (Student 16, Welch 17). With cell QC applied, fig4-glp2r-sag is not reproduced
(sag BH p 0.060), edtable1-species-ephys methods underspecified (Student 16, Welch 17) and
fig6-deep-ephys-anova partially reproduced (12 and 4). fig5c-frem3-ephys-depth, fig6c-carm1p1-upstroke-rin
and fig6c-col22a1-rin-fi-sag are the same under every set. The classifier gives 0.612 to 0.642 over 50
splits under the three sets run, within 0.67 +/- 0.067.


## Claims on data outside the dandiset (`elsewhere`)

These use tables and images from the paper's code repository at commit 3daf9f0, not the dandiset.
They are counted apart from the dataset's claims.

### fig2d-pathology-scores and edfig1-marker-correlation

> Average pathology scores over six markers ranged from 0.06 to 1.83 across donors, and most cases
> had averages below 1.0.
>
> The six pathology markers are very weakly correlated with one another across donors, with only
> Ki-67 and Nissl (cellularity) modestly correlated.

In [ ]:
SIX = ["GFAP Cortex", "Iba1 Cortex", "Ki67", "SMI32 Cortex", "NeuN Orientation Layering", "Cellularity"]
avg = path_tab[SIX].sum(axis=1, min_count=6) / 6
print(f"donors with all six scores: {avg.notna().sum()} of {len(path_tab)}; average {avg.min():.3f} to {avg.max():.3f}; "
      f"below 1.0: {(avg < 1).sum()} ({(avg < 1).mean():.1%} of those with all six)")

**Result: reproduced.** Over the 86 donors with all six scores, the average ranges from 0.056 to
1.833, within the tolerances of 0.06 and 1.83, and 78 (86.7%) are below 1.0.


In [ ]:
cor = path_tab[SIX].corr(method="pearson")
pairs = pd.DataFrame([dict(a=a, b=b, r=cor.loc[a, b], n=path_tab[[a, b]].dropna().shape[0]) for a, b in combinations(SIX, 2)])
pairs["abs_r"] = pairs.r.abs()
print(pairs.sort_values("abs_r", ascending=False).round(3).to_string(index=False))

**Result: reproduced.** Ki-67 and cellularity have the largest absolute correlation of the 15 pairs
(r = 0.391); the next is 0.254.


### Morphological features: fig4f-morph-anova, fig5b-frem3-morph-depth, fig6-deep-morph-anova

In [ ]:
MF = list(morph_names.iloc[:, 1])
mh = morph_h.set_index("specimen_id").join(tab[["SeuratMapping", "L23_depth_normalized"]], how="inner")
mh["tt"], mh["nd"] = mh.SeuratMapping.str.split().str[-1], mh.L23_depth_normalized
mm_ = morph_m.set_index("specimen_id").join(mt[["tt"]], how="inner")
print("human reconstructions with a t-type:", len(mh), mh.tt.value_counts().to_dict(), "| mouse:", len(mm_), mm_.tt.value_counts().to_dict())
res_mh = anova(mh[mh.tt.isin(HOM)], "tt", MF)
res_mm = anova(mm_, "tt", MF)
# as the code: a feature with no p-value (constant among the cells) is dropped before BH, and R2 is counted over the rest
print("dropped, no p-value: human", list(res_mh.feature[res_mh.p.isna()]), "| mouse", list(res_mm.feature[res_mm.p.isna()]))
big_h, big_m = res_mh[(res_mh.r2 > 0.15) & res_mh.p.notna()], res_mm[(res_mm.r2 > 0.15) & res_mm.p.notna()]
print(f"human, all FREM3: R2 > 0.15 in {len(big_h)} of {int(res_mh.p.notna().sum())} kept (expected 16 +/- 1.6), of them FDR < 1e-3: {(big_h.p_bh < 1e-3).sum()}")
print(f"mouse: R2 > 0.15 in {len(big_m)} of {int(res_mm.p.notna().sum())} kept (expected 12 +/- 1.2), of them FDR < 0.05: {(big_m.p_bh < 0.05).sum()} (paper 10)")
res_mh_sup = anova(mh[mh.tt.isin(["LTK", "GLP2R"]) | ((mh.tt == "FREM3") & (mh.nd < 0.575))], "tt", MF)
print(f"variant, superficial FREM3: R2 > 0.15 in {((res_mh_sup.r2 > 0.15) & res_mh_sup.p.notna()).sum()}, of them FDR < 1e-3: "
      f"{((res_mh_sup.r2 > 0.15) & (res_mh_sup.p_bh < 1e-3)).sum()}")

**Result: reproduced.** Among the human homologous types two features have no p-value and are
dropped, as in the code; 16 of the 58 kept have R2 above 0.15, all 16 at BH p below 1e-3. Among the
mouse types one is dropped, and 12 of the 59 kept have R2 above 0.15, 10 of them at BH p below 0.05,
as the paper states. With superficial FREM3 only (the code's revision), 10 human features exceed
0.15.


In [ ]:
LOC = ["soma_relative_depth_none", "depth_from_pia"]
fm = mh[(mh.tt == "FREM3") & mh.nd.notna()]
r5 = depth_regression(fm, MF)
kept = r5[r5.p.notna()]
print(f"FREM3 reconstructions with a depth: {len(fm)}; features {len(MF)}")
print("dropped before BH, no p-value:", list(r5.feature[r5.p.isna()]),
      "| constant among these cells:", [f for f in MF if fm[f].nunique() <= 1])
print(f"over the {len(kept)} kept: FDR < 0.05 {(kept.p_bh < 0.05).sum()} (expected 37 +/- 3.7), "
      f"< 0.01 {(kept.p_bh < 0.01).sum()} (expected 28 +/- 2.8)")
r_loc = depth_regression(fm, [f for f in MF if f not in LOC])
print(f"for reference, the earlier reading without the two location features: {int(r_loc.p.notna().sum())} kept, "
      f"FDR < 0.05 {(r_loc.p_bh < 0.05).sum()}, < 0.01 {(r_loc.p_bh < 0.01).sum()}")
print(r5.set_index("feature").loc[["apical_dendrite_extent_y", "basal_dendrite_max_path_distance_none", "soma_radius"],
                                  ["n", "r", "r2", "p_bh"]].round(6).to_string())


**Result: reproduced.** Of the 60 features, two are constant among the 56 FREM3 reconstructions with
a depth and have no p-value; over the 58 kept, as in the code, 37 correlate with depth at BH p below
0.05 and 28 below 0.01, the paper's counts. Apical height (r = 0.94), maximum basal path distance
(0.74) and soma radius (0.49, BH p 0.018) are among them. Without the two location features, the
earlier reading, the counts are 35 and 26.


In [ ]:
mh["deep_group"] = np.where((mh.tt == "FREM3") & (mh.nd > 0.575), "deep FREM3",
                            np.where(mh.tt.isin(["CARM1P1", "COL22A1"]), mh.tt, None))
dm_ = mh[mh.deep_group.notna()]
res_dm = anova(dm_, "deep_group", MF)
print("reconstructions per deep group:", dm_.deep_group.value_counts().to_dict())
print("dropped before BH, no p-value:", list(res_dm.feature[res_dm.p.isna()]))
print(f"over the {int(res_dm.p.notna().sum())} kept: FDR < 0.05: {(res_dm.p_bh < 0.05).sum()} (expected 24 +/- 2.4); "
      f"R2 > 0.4: {((res_dm.r2 > 0.4) & res_dm.p.notna()).sum()} (expected 20 +/- 2)")

**Result: reproduced.** One feature has no p-value among the deep types and is dropped, as in the
code; of the 59 kept, 24 differ among the deep types at BH p below 0.05 and 20 have R2 above 0.4,
with 8 deep FREM3, 5 CARM1P1 and 10 COL22A1 reconstructions.


### fig6a-deep-dendrites

> CARM1P1 neurons have the largest total dendritic length of the L2 and L3 t-types, with extensive
> proximal oblique and basal branching, while COL22A1 neurons have very sparse branching.

The criterion, as updated after the question, gives the readings the paper allows: total length is
apical plus basal total length; the t-types are (a) the five with FREM3 whole or (b) with FREM3 split
at 0.575; largest is by (i) the mean or (ii) the median, with no test. Sparse branching is the fewest
apical and basal branches among the three deep types. The outcome must hold under every reading.


In [ ]:
mh["total_length"] = mh.apical_dendrite_total_length_none + mh.basal_dendrite_total_length_none
mh["group_b"] = np.where(mh.tt == "FREM3", np.where(mh.nd > 0.575, "deep FREM3", np.where(mh.nd.notna(), "superficial FREM3", None)), mh.tt)
BR = ["apical_dendrite_num_branches_none", "basal_dendrite_num_branches_none"]
cols = ["total_length", "apical_dendrite_total_length_none"] + BR
print("reconstructions without a depth (left out of reading (b)):", mh[mh.tt == "FREM3"].nd.isna().sum(), "FREM3")
fig6a = {}
for gname, g in [("(a) FREM3 whole", "tt"), ("(b) FREM3 split at 0.575", "group_b")]:
    agg = mh.dropna(subset=[g]).groupby(g)[cols].agg(["count", "mean", "median"])
    print(f"\n{gname}:")
    print(agg.round(0).to_string())
    for stat in ("mean", "median"):
        tl = agg[("total_length", stat)]
        fig6a[(gname, stat, "CARM1P1 largest total length")] = tl.idxmax() == "CARM1P1"
        print(f"  {stat}: largest total length {tl.idxmax()} ({tl.max():.0f} um), next {tl.drop(tl.idxmax()).idxmax()} ({tl.drop(tl.idxmax()).max():.0f} um)")
dg = mh[mh.deep_group.notna()].groupby("deep_group")[["apical_dendrite_total_length_none"] + BR].agg(["mean", "median"])
print("\ndeep types:")
print(dg.round(1).to_string())
for stat in ("mean", "median"):
    fig6a[("deep types", stat, "COL22A1 fewest apical and basal branches")] = all(dg[(b, stat)].idxmin() == "COL22A1" for b in BR)
    fig6a[("deep types", stat, "CARM1P1 apical length below deep FREM3 (not judged)")] = \
        dg.loc["CARM1P1", ("apical_dendrite_total_length_none", stat)] < dg.loc["deep FREM3", ("apical_dendrite_total_length_none", stat)]
print()
for k, v in fig6a.items():
    print(" | ".join(k), ":", v)
judged = [v for k, v in fig6a.items() if "not judged" not in k[2]]
print("\nCARM1P1 largest and COL22A1 sparsest under every reading:", all(judged), "| under some:", any(judged))


**Result: blocked, methods underspecified (the readings differ).** With FREM3 whole, CARM1P1 has the
largest total dendritic length by mean (12011 um) and median (10447 um). With FREM3 split at 0.575,
CARM1P1 is largest by mean (12011 um against 11566 for deep FREM3) but deep FREM3 is largest by median
(12024 um against 10447). COL22A1 has the fewest apical and basal branches of the deep types by mean
and median (24 and 19 by mean, against at least 56 and 59). The apical part does not hold: CARM1P1's
apical length (mean 6896 um) is above that of deep FREM3 (5443 um).


### edfig6-classifier-morphoelectric

> Combining morphological and electrophysiological features raises accuracy slightly, to 69% for
> human and 60% for mouse neurons.

The method of edfig6-classifier-human-ephys, as the updated criterion says, on the homologous cells of
each species with all 18 electrophysiological features and a reconstruction, with the 18 and the 60
morphological features side by side. The main test uses the code's tables, as the criterion names
them (72 human and 50 mouse cells); as a variant, the human electrophysiological features are those
computed here from the dandiset (reading M2 of the link). Each is run on 250 splits, not 1000, to
keep the notebook within 30 minutes; the standard error of the mean is printed.


In [ ]:
t = time.time()
mf_h = morph_h.set_index("specimen_id")[MF]
hom_code = tab.loc[tab.index.isin(etab.index)].assign(tt=lambda x: x.SeuratMapping.str.split().str[-1])
hom_code = hom_code[hom_code.tt.isin(HOM)][["tt"]].join(etab[F]).join(mf_h, how="inner")
mo_me = mo[["tt"] + F].join(morph_m.set_index("specimen_id")[MF], how="inner")
hom_here = hom[hom.specimen_id.isin(mf_h.index)].set_index("specimen_id")[["tt"] + F].join(mf_h, how="inner")
me = {}
for name, d, n in [("human, code's tables", hom_code, 250), ("mouse, code's tables", mo_me, 250),
                   ("variant: human, features computed here", hom_here, 100)]:
    dd = d[d[F].notna().all(axis=1)]
    print(f"{name}: {len(dd)} cells {dd.tt.value_counts().to_dict()}, missing morphological values "
          f"{int(dd[MF].isna().sum().sum())}")
    acc, _ = rf_accuracy(d, "edfig6-classifier-morphoelectric-" + name, n_splits=n, feats=F + MF)
    acc_e, _ = rf_accuracy(d, "edfig6-classifier-morphoelectric-ephys-" + name, n_splits=100)
    me[name] = acc.mean()
    print(f"  combined: mean accuracy over {n} splits {acc.mean():.4f} (SE {acc.std() / np.sqrt(n):.4f}, SD {acc.std():.4f}); "
          f"electrophysiology alone on the same cells, 100 splits, {acc_e.mean():.4f}; largest class {dd.tt.value_counts().max() / len(dd):.3f}")
print({k: round(v, 3) for k, v in me.items()}, "| expected human 0.69 +/- 0.069, mouse 0.60 +/- 0.06")
print(f"{(time.time() - t) / 60:.1f} min")


**Result: reproduced.** On the code's tables, the 72 human cells (42 FREM3, 20 LTK, 10 GLP2R) give
a mean accuracy of 0.720 over 250 splits (SE 0.005), within 0.69 +/- 0.069, and the 50 mouse cells
give 0.582 (SE 0.007), within 0.60 +/- 0.06. On the same cells the electrophysiological features
alone give 0.679 (human) and 0.527 (mouse) over 100 splits, so the combined features raise accuracy,
as stated. With the human features computed here (70 cells), the combined accuracy is 0.715 over 100 splits.

### fig5d-frem3-gene-depth

> FREM3 gene expression varies with soma depth; 790 genes correlate with depth at FDR below 0.05.

In [ ]:
print("genes in the table:", len(genes_depth), "| FDR < 0.05:", int((genes_depth.p_corr < 0.05).sum()),
      "(expected 790 +/- 79) | FDR < 0.01:", int((genes_depth.p_corr < 0.01).sum()))
print("p_corr range", genes_depth.p_corr.min(), "to", genes_depth.p_corr.max(), "| features", genes_depth.feature.unique())

**Result: blocked (other).** The code's table gives 4417 genes at FDR below 0.05 (against 790) and
1917 below 0.01, the count the code's own notebook gives for its revision; the preprint's gene
results are not public, so the claim cannot be tested on its own data.


### Figure 1a-c: NeuN sections

A port of the code's Code6 (human) and Code7 (mouse) R scripts to Python (`patchseq_l23.section_tables`).
The scripts' comments print values the port can be checked against: the five human L2/3 thicknesses
(1058, 1293, 1387, 1073 and 1344 um), the mouse L2/3 thicknesses (281, 223 and 257 um), the L2 fractions
of L2/3 (human -0.179 to -0.199, mouse -0.246, -0.205 and -0.325), the densities (human 27.7 +/- 4.46,
mouse 165 +/- 24.9 thousand per mm3) and the pooled quantiles of soma area. Code7 as committed reads the
four mouse lines as L2, L4, L5 and L6 and sets 100 um = 330 pixels for every mouse section; that does
not give its printed values. Reading the lines as L1 to L4, as Code6 does for human, with case_set_1 at
100 um = 165 pixels (the scale Code7's own comment gives), gives every printed value, and is used. Both
readings are printed.

In [ ]:
H = P.section_tables("human")
M = P.section_tables("mouse")  # the reading that gives Code7's printed values
M_committed = P.section_tables("mouse", "as_committed")
for sp_name, tabs in [("human", H), ("mouse", M), ("mouse, Code7 as committed", M_committed)]:
    for case, t in tabs.items():
        c = t["cells"]
        print(f"{sp_name} {case}: {t['lines']} boundary lines, {len(c)} somata, L2/3 thickness {t['l23_um']:.0f} um, "
              f"L2 fraction {t['l2_fraction']:.4f}, {100 / t['um_per_px']:.0f} pixels per 100 um, "
              f"scaled depth before clipping {c.scaled_unclipped.min():.2f} to {c.scaled_unclipped.max():.2f} "
              f"({int((c.scaled_unclipped < -1).sum())} below -1, {int((c.scaled_unclipped > 0).sum())} above 0), "
              f"area {c.area.min():.1f} to {c.area.max():.1f} um2")
dens_h, per_h = P.density_profile(H)
dens_m, per_m = P.density_profile(M)
_, per_mc = P.density_profile(M_committed)
am_c = pd.concat([v["cells"].area for v in M_committed.values()])
print(f"\nmouse, Code7 as committed: density {per_mc.mean():.1f} +/- {per_mc.std():.1f} thousand per mm3, pooled area quartiles "
      f"{am_c.quantile(0.25):.1f} and {am_c.quantile(0.75):.1f} um2 (the first round of this attempt)")

#### fig1a-thickness

In [ ]:
l23 = pd.Series({k: v["l23_um"] for k, v in H.items()}) / 1000
print("human L2+L3 thickness per section (um):", (l23 * 1000).round(0).to_dict())
print(f"human L2+L3 thickness: {l23.mean():.3f} +/- {l23.std():.3f} mm over {len(l23)} donors (expected 1.23 +/- 0.15); "
      f"within tolerance: {abs(l23.mean() - 1.23) <= 0.15}")
print("mouse: the sections carry", sorted(set(v["lines"] for v in M.values())),
      "lines, the deepest the L3/L4 border, with no white matter line, so the whole cortex (1.06 mm) is not tested; "
      "mouse L2/3 thickness, reported only (Code7 prints 281, 223 and 257):", {k: round(v["l23_um"]) for k, v in M.items()}, "um")

**Result: reproduced, for the human value; the mouse value is untested.** The human L2 plus L3
thickness is 1.231 +/- 0.155 mm over 5 sections, within 1.23 +/- 0.15. The mouse sections end at the
L3/L4 border, with no white matter line, so the thickness of the whole mouse cortex (1.06 mm) and the
ratio of 1.16 are not tested, as the criterion says. The mouse L2/3 thickness, reported only, is 281,
223 and 257 um, the values Code7 prints.

#### fig1b-density

In [ ]:
print(f"human: {per_h.mean():.2f} +/- {per_h.std():.2f} thousand cells/mm3 (expected 27.7 +/- 4.5); per section {per_h.round(1).to_dict()}")
print(f"mouse: {per_m.mean():.2f} +/- {per_m.std():.2f} thousand cells/mm3 (expected 165 +/- 24.9); per section {per_m.round(1).to_dict()}")
prof = pd.concat([dens_h.assign(species="human"), dens_m.assign(species="mouse")])
prof = prof[(prof.bin > 0) & (prof.bin < 19)]  # first and last bins dropped, as the code
pm = prof.groupby(["species", "centre"]).dens.mean().unstack(0).sort_index(ascending=False)
print("\nmean density by bin (0 = L1/L2 border, -1 = L3/L4 border), first and last bins dropped:")
print(pm.round(3).to_string())
hp = pm.human
print(f"human: highest at scaled depth {hp.idxmax():.3f} ({hp.max():.3f}), lowest at {hp.idxmin():.3f} ({hp.min():.3f}); "
      f"ratio {hp.min() / hp.max():.2f}; mouse ratio of lowest to highest {pm.mouse.min() / pm.mouse.max():.2f}")
mp_ = pm.mouse
print(f"mouse: lowest bin over highest {mp_.min() / mp_.max():.2f}, coefficient of variation over bins {mp_.std() / mp_.mean():.3f} "
      f"(human {hp.std() / hp.mean():.3f})")

**Result: reproduced.** The human density is 27.70 +/- 4.46 thousand cells/mm3 and the mouse density
164.74 +/- 24.91, both within tolerance and as Code7 prints them. Human density is highest just below
L1 (1.742) and lowest at scaled depth -0.575 (0.745), a ratio of 0.43, about half. Mouse density varies
less with depth: its lowest bin is 0.74 of its highest, and its coefficient of variation over bins is
0.080, against 0.282 in human. Code7 as committed (lines read as L2, L4, L5 and L6, and 330 pixels per
100 um in every mouse section) gives a mouse density of 349.8 +/- 338.5, the first round's value, and
puts every mouse soma below the deepest line it uses.

#### fig1c-soma-area and fig1c-soma-variation

In [ ]:
ah = pd.concat([v["cells"].area for v in H.values()])
am = pd.concat([v["cells"].area for v in M.values()])
q = pd.DataFrame({"human": ah.quantile([0.25, 0.75]), "mouse": am.quantile([0.25, 0.75])})
print("pooled quartiles over all somata of a species (um2), as judged:\n", q.round(1).to_string(), f"\nlargest human soma {ah.max():.1f} um2; n human {len(ah)}, mouse {len(am)}")
qs = [0, 0.05, 0.25, 0.5, 0.75, 0.9, 0.99, 1]
print("pooled quantiles", qs, "\n  human", ah.quantile(qs).round(2).tolist(), "\n  mouse", am.quantile(qs).round(2).tolist())
per_case = pd.DataFrame({f"{sp_}:{k}": v["cells"].area.quantile([0.25, 0.75]) for sp_, tabs in [("human", H), ("mouse", M)] for k, v in tabs.items()}).T
print("\nper section, reported only:\n", per_case.round(1).to_string())
exp_q = {("human", 0.25): (107, 10.7), ("human", 0.75): (253, 25.3), ("mouse", 0.25): (44, 4.4), ("mouse", 0.75): (96, 9.6)}
print("within tolerance:", {f"{s_} {qq}": bool(abs(q.loc[qq, s_] - v) <= tol) for (s_, qq), (v, tol) in exp_q.items()},
      "| largest human soma above 850 um2:", ah.max() > 850)
print("mean of section quartiles: human", per_case.filter(like="human", axis=0).mean().round(1).to_dict(),
      "mouse", per_case.filter(like="mouse", axis=0).mean().round(1).to_dict())
pa = prof.groupby(["species", "centre"]).area.mean().unstack(0).sort_index(ascending=False)
print("\nmean area by bin (first and last dropped):\n", pa.round(1).to_string())
print(f"deepest kept bin over shallowest: human {pa.human.iloc[-1] / pa.human.iloc[0]:.2f}, mouse {pa.mouse.iloc[-1] / pa.mouse.iloc[0]:.2f}")

**Result: reproduced.** Pooled over all somata of a species, as the criterion says, the human quartiles
are 107.5 and 252.6 um2 and the mouse quartiles 44.7 and 96.5 um2, all within tolerance and equal to
the quantiles Code7 prints, and the largest human soma is 886.4 um2, above 850. Per section, reported
only, the means of section quartiles are 110.8 and 254.1 (human) and 50.1 and 91.8 (mouse). In human
the mean area rises from 127.0 um2 in the shallowest kept bin to 245.4 at its peak; in mouse it ranges
from 59.3 to 81.5 um2, and the deepest kept bin is 1.26 times the shallowest.

In [ ]:
allb = pd.concat([dens_h.assign(species="human"), dens_m.assign(species="mouse")])
for name, sel in [("(D1, judged) Code7's bins, centres -0.95 to -0.7", (allb.centre >= -0.95) & (allb.centre < -0.7)),
                  ("(D2, variant) centres deeper than -0.575, deepest bin dropped", (allb.centre < -0.575) & (allb.bin > 0))]:
    print(name, "| bin centres", sorted(allb[sel].centre.round(3).unique()))
    het = allb[sel].groupby("species").het.mean()
    nm = int(allb[sel & (allb.species == "mouse")].n.sum())
    nh = int(allb[sel & (allb.species == "human")].n.sum())
    ratio = het.human / het.mouse if np.isfinite(het.get("mouse", np.nan)) else np.nan
    print(f"  SD human {het.human:.1f} um2 ({nh} somata), mouse {het.get('mouse', np.nan):.1f} um2 ({nm} somata); "
          f"ratio {ratio:.2f} (expected 4 +/- 0.4)")
print("bins: 0 is the deepest (centre -0.975)")
# variant: the areas binned as the density is (right-closed), not as the scripts bin them (left-closed)
dh_r, _ = P.density_profile(H, area_bins="right")
dm_r, _ = P.density_profile(M, area_bins="right")
allr = pd.concat([dh_r.assign(species="human"), dm_r.assign(species="mouse")])
het_r = allr[(allr.centre >= -0.95) & (allr.centre < -0.7)].groupby("species").het.mean()
print(f"variant, right-closed area bins: D1 ratio {het_r.human / het_r.mouse:.2f}; somata at scaled depth exactly 0 "
      f"(in no left-closed area bin): human {sum(int((v['cells'].scaled_depth == 0).sum()) for v in H.values())}, "
      f"mouse {sum(int((v['cells'].scaled_depth == 0).sum()) for v in M.values())}")
sd_m = pd.concat([v["cells"].scaled_depth for v in M.values()])
print(f"mouse somata: {len(sd_m)}, scaled depth {sd_m.min():.2f} to {sd_m.max():.2f}, {int((sd_m == -1).sum())} clipped to -1")
het2 = allb[allb.centre >= -0.15].groupby("species").het.mean()
print(f"for reference, L2 bins (centres from -0.15): ratio {het2.human / het2.mouse:.2f}")

**Result: reproduced.** With deep L3 as Code7's five bins (D1, as the criterion says), the mean SD of
soma area is 117.6 um2 in human (948 somata) and 29.8 um2 in mouse (357 somata), a ratio of 3.94,
within 4 +/- 0.4. The variant D2 gives 3.83, and binning the areas as the density is binned gives 3.94.
Five human somata sit at scaled depth exactly 0 and fall in no area bin, as in the scripts; none is in
deep L3.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| features-and-labels-link | with labels | reproduced under M2 (M1 links no cell) |
| cohort-cells | with labels | not reproduced (318 cells, 280 with features) |
| cohort-donors | with labels | not reproduced (56 donors) |
| cohort-type-counts | with labels | not reproduced |
| fig3d-depth-by-type | with labels | partially reproduced |
| fig4e-human-ephys-anova | with labels | partially reproduced |
| fig4-glp2r-sag | with labels | reproduced |
| edfig6-classifier-human-ephys | with labels | reproduced |
| fig5c-frem3-ephys-depth | with labels | partially reproduced |
| fig6-deep-ephys-anova | with labels | reproduced |
| fig6c-carm1p1-upstroke-rin | with labels | blocked, methods underspecified |
| fig6c-col22a1-rin-fi-sag | with labels | reproduced |
| fig4e-mouse-ephys-anova | with inputs | reproduced |
| edtable1-species-ephys | with inputs | reproduced |
| edfig6-classifier-mouse-ephys | with inputs | reproduced |
| fig2ef-pathology-ephys | with inputs | partially reproduced |
| fig2d-pathology-scores | elsewhere | reproduced |
| edfig1-marker-correlation | elsewhere | reproduced |
| fig4f-morph-anova | elsewhere | reproduced |
| fig5b-frem3-morph-depth | elsewhere | reproduced |
| fig6-deep-morph-anova | elsewhere | reproduced |
| fig1a-thickness | elsewhere | reproduced (human value; mouse value untested) |
| fig1b-density | elsewhere | reproduced |
| fig1c-soma-area | elsewhere | reproduced |
| fig1c-soma-variation | elsewhere | reproduced |
| fig5d-frem3-gene-depth | elsewhere | blocked (revision's table) |
| edfig6-classifier-morphoelectric | elsewhere | reproduced |
| fig6a-deep-dendrites | elsewhere | blocked, methods underspecified (readings differ) |
| fig1f-frem3-heterogeneity, fig1g-deep-types-de-genes, fig3a-platform-correlation, fig6e-deep-de-genes, fig6f-smi32 | elsewhere | blocked (data not pinned or not public) |

How the analysis differs from the paper's: The files carry no SpecimenID; cells were matched
to the code's table through the session id (the next SpecimenID of the same donor, reading M2 of
the link), which the features confirm for the 283 cells with features in the code and the sample
dates and tube order support for the other 35; under M1, a SpecimenID stored in the files, no cell
links. Features were computed with IPFX 2.1.2 (the paper gives no version) and read from its output as
the code's feature names imply. The sweep QC is the Methods', with the bias current and bridge balance
limits that IPFX 2.1.2 does not apply added, and the input resistance of IPFX's cell QC for the 15%
limit, a choice the Methods do not fix; IPFX's sweep QC alone matches the code's table more closely. No
cell QC is applied, as the claims file asks. The analysis cells are every file whose matched cell has
the feature, which adds 19 cells to the code's 283 and loses 22 of them; the section on cell sets
gives every electrophysiology outcome on the 283 alone, with cell QC and with IPFX's sweep QC. Labels,
pathology scores and the mouse features come from the code's tables, which are from the paper's
revision. Code6 and Code7 were ported to Python; for mouse, the port reads the boundary lines and pixel
scale in the way that gives Code7's printed values, not as Code7's code is committed. The
morpho-electric classifier runs 250 splits, not 1000, to keep the notebook within 30 minutes.

What was found about the dataset: It holds 318 recordings from 56 donors: every one of the
283 cells with features in the paper's code, and 35 more of the code's 385 cells; 67 cells of the
table are not in it. The files carry no SpecimenID, t-type, depth or region, so no claim of the
paper can be tested from the dandiset alone, and the archive's description does not say which
cells were deposited or why. Donor H18.03.318 is female in the files and male in the paper's donor
table, and donor H18.06.005 is 56 years old in the files and 57 in the table. The recordings
themselves are consistent with the paper's code: spike times computed here equal those stored in
the files, and the features equal the code's values in most cells. The code's adaptation is
the mean over all spiking long squares and its initial rate and tau come from the hero sweep and
long squares, where the Methods describe adaptation at about 50 pA above rheobase, initial rate at
rheobase and tau from short pulses. In the code repository, Code7 as committed does not give the
values its comments print; the reading that does takes the mouse lines as Code6 takes the human ones
and case_set_1 at 165 pixels per 100 um.


In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")